In [ ]:
import os, sys, pathlib, subprocess, json, urllib.request, shutil, builtins, runpy
from unittest.mock import patch
import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), 'STOP: GPU not connected'
repo = pathlib.Path('/content/Reproducing-research-')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/KanakMalpani/Reproducing-research-.git',str(repo)],check=True)
audit = repo/'clear-cloudy-gap-audit'
os.chdir(audit)
results = audit/'results'
results.mkdir(exist_ok=True)
real_open = builtins.open
report = []
for script, output in [('kill_test.py','kill_test_results.json'),('ndvi_comatch_analyze.py','ndvi_comatch_results.json'),('p2_analyze.py','p2_analyze.out.json')]:
    def safe_open(file,mode='r',*args,**kwargs):
        if isinstance(file,(str,os.PathLike)) and any(c in mode for c in 'wax+'):
            assert pathlib.Path(file).name == output
            file = results/output
        return real_open(file,mode,*args,**kwargs)
    with patch('builtins.open',safe_open):
        runpy.run_path(script,run_name='__main__')
    equal = json.loads((audit/output).read_text()) == json.loads((results/output).read_text())
    report.append({'script':script,'exact_json_match':equal})
    print('REFERENCE CHECK',script,equal)
    assert equal, 'STOP: reference mismatch'
(results/'reproduction_checks.json').write_text(json.dumps(report,indent=2))
print('ALL THREE REFERENCE CHECKS PASSED')

GPU: True Tesla T4
val {"scenes": 3, "mean_gap_pts": 47.32278115484655, "median_gap_pts": 36.61608824132029, "ci95": [19.56356370738137, 85.78869151583801], "frac_scenes_clear_better": 1.0, "verdict": "PASS (headroom >=5 pts)"} | visible-px: null
test {"scenes": 5, "mean_gap_pts": 34.542337110160055, "median_gap_pts": 36.18039703859902, "ci95": [14.646080275068346, 54.43859394525175], "frac_scenes_clear_better": 1.0, "verdict": "PASS (headroom >=5 pts)"} | visible-px: null
test_val {"scenes": 8, "mean_gap_pts": 39.33500362691749, "median_gap_pts": 36.398242639959655, "ci95": [22.442475796323194, 57.577351767791185], "frac_scenes_clear_better": 1.0, "verdict": "PASS (headroom >=5 pts)"} | visible-px: {"scenes": 8, "mean_gap_pts": 32.68198790410891, "median_gap_pts": 24.64197618829039, "ci95": [15.626223512754686, 52.38714936519628], "frac_scenes_clear_better": 1.0, "verdict": "PASS (headroom >=5 pts)"}
REFERENCE CHECK kill_test.py True
water_only n 7 [np.float64(18.55), np.float64(-7.75

In [22]:
exec("import pathlib,os,json,shutil,zipfile,hashlib,datetime\nfrom google.colab import files\nrepo=pathlib.Path('/content/Reproducing-research-');audit=repo/'clear-cloudy-gap-audit';results=audit/'results'\nassert audit.exists(), 'STOP missing audit repository'\nR=results/'finetune_v1'\ncomparison=json.loads((R/'candidate_comparison.json').read_text());selection=json.loads((R/'selection.json').read_text())\ntext=['# Work log and handover','', 'Model improvement stopped at the user request on 2026-11-15. No second candidate was trained. Released WF2 weights remain frozen.','', '## Frozen audit','Reproductions matched committed JSON and GPU confusion counts. Corrected pooled medians, noncloud placebo, JRC known-code strata, brightness .3/.35, separate Campo Galliano and Ituango South, and frozen Prithvi comparison completed. See results/RUN_REPORT.md and result JSON.','', '## Separate authorized fine-tune',f\"Six epochs; epoch {selection['selected']['epoch']} selected by validation only. Train388 scenes, 38596 tiles, excluding evaluation EMSR activations. Validation water IoU 0.8651451103384014 to 0.9162587023504096. Test pooled water IoU {comparison['comparison']['test']['all']['baseline_iou']} to {comparison['comparison']['test']['all']['candidate_iou']}. 90% test target not met. Cloudy test IoU decreased from 0.7667548616804217 to 0.7390338550118456. Paired mean scene change -0.4026345468267597 points, 95% CI [-5.988669374462191,3.8530393481277763]. This does not establish a general improvement.\",'Threshold calibration considered .20 through .90 on validation only; selected .50, bias shift zero. Weights and full optimizer/training state preserved. Existing test was previously inspected, not a fresh blind benchmark.','', '## Remaining audit limitation','JRC code0 has no observations; previous known nonpermanent {1,2} metric is not confirmed flood water. A completion follow-up must report historically nonwater code1 separately from seasonal code2 and quantify coverage. No unknown pixel should silently become flood.','', '## Preservation','Archive contains repository source, rules, result JSON, model assets, trained checkpoints and optimizer state. External raw rasters and rebuildable .venv caches are not part of this repository archive; dataset/checksum manifests and environment requirements preserve retrieval information.']\n(audit/'WORK_LOG.md').write_text(chr(10).join(text)+chr(10))\nreleased=audit/'checkpoints'/'released_frozen';released.mkdir(parents=True,exist_ok=True)\nsource=pathlib.Path(os.environ['WF_CKPT']);assert source.exists()\nshutil.copy2(source,released/'model.pt')\nassert hashlib.sha256((released/'model.pt').read_bytes()).hexdigest()==selection['released_checkpoint_sha256']\n(results/'preservation_inventory.json').write_text(json.dumps(dict(date='2026-11-15',model_improvement='stopped',raw_data_root=os.environ['WF_ROOT'],raw_data_in_archive=False,environment_cache_in_archive=False,files=[dict(path=str(p.relative_to(repo)),bytes=p.stat().st_size) for p in repo.rglob('*') if p.is_file() and not any(x in {'.venv-prithvi','__pycache__'} for x in p.parts)]),indent=2))\npackage=pathlib.Path('/content/worldfloods-work-preserved.zip')\nwith zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED,compresslevel=1) as z:\n for p in repo.rglob('*'):\n  if p.is_file() and not any(x in {'.venv-prithvi','__pycache__'} for x in p.parts):z.write(p,str(pathlib.Path('Reproducing-research-')/p.relative_to(repo)))\nprint('PRESERVED ARCHIVE READY',package.stat().st_size,flush=True)\nfiles.download(str(package))\n")

PRESERVED ARCHIVE READY 1389286112


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
exec("(audit/'permanent_flood_completion.py').write_text(\"\\\"\\\"\\\"Complete the supplied JRC stratification with separate code1 flood proxy and code2 seasonal water.\\\"\\\"\\\"\\nimport os,sys,pathlib,json,hashlib,numpy as np,torch,rasterio\\nfrom rasterio.windows import Window\\nA=pathlib.Path(__file__).resolve().parent;os.chdir(A);sys.path.insert(0,str(A))\\nfrom wf_data import scenes,NORM\\nR=A/'results';ROOT=pathlib.Path(os.environ['WF_ROOT']);BASE=pathlib.Path(os.environ['WF_CKPT'])\\nrule='Frozen released WF2. Original256px grid stride512 and >=50% valid GT. JRC YearlyHistory codes0=no observations,1=not water,2=seasonal,3=permanent. Separate binary-water IoUs on code3 permanent, code1 historically-nonwater flood proxy, code2 seasonal, code0 unknown; never label code0 flood. All/clear/cloud pixels reported. Pool confusion within scene; unweighted scene means with >=200 GT-water gate and5000 scene-bootstrap draws seed42. Also report secondary pooled IoU and GT-water coverage. JRC is used to build released GT, so permanent-water scores are label-dependent. Nonpermanent water is not independently confirmed flood; whole-image genuine flood split remains unidentifiable without labels for unknown regions.'\\nassert BASE.exists() and ROOT.exists()\\nfor sp in ('val','test'):\\n for name,s2,gt in scenes(sp):assert pathlib.Path(s2).exists() and pathlib.Path(gt).exists() and (ROOT/sp/'PERMANENTWATERJRC'/(name+'.tif')).exists(), 'STOP missing required path'\\n(A/'PERMANENT_FLOOD_COMPLETION_RULE.md').write_text(rule+chr(10)+'Sources: https://developers.google.com/earth-engine/datasets/catalog/JRC_GSW1_4_YearlyHistory and ml4floods/data/create_gt.py, ee_download.py commit192426c7286315b59a2f52c7a9cdebe589242de2.')\\nns={};exec((A/'cloud_iou_pilot.py').read_text().split('BINS=')[0],ns);m=ns['m'].cuda().eval()\\nbefore=hashlib.sha256(BASE.read_bytes()).hexdigest();torch.set_num_threads(3)\\ndef cm(p,w,v):return np.array([(p&w&v).sum(),(p&~w&v).sum(),(~p&w&v).sum()],np.int64)\\ndef iou(x):return float(x[0]/sum(x)) if sum(x) else None\\nnames={0:'unknown',1:'flood_proxy_historically_nonwater',2:'seasonal',3:'permanent'};rows=[]\\nfor sp in ('val','test'):\\n for name,s2,gt in scenes(sp):\\n  counts={kind:{region:np.zeros(3,np.int64) for region in ('all','clear','cloud')} for kind in names.values()};coverage={kind:dict(valid_pixels=0,gt_water_pixels=0) for kind in names.values()};n=0\\n  with rasterio.open(s2) as S,rasterio.open(gt) as G,rasterio.open(ROOT/sp/'PERMANENTWATERJRC'/(name+'.tif')) as J:\\n   assert S.shape==G.shape==J.shape and S.transform==G.transform==J.transform and S.crs==G.crs==J.crs\\n   for r in range(0,G.height-255,512):\\n    for c in range(0,G.width-255,512):\\n     win=Window(c,r,256,256);g=G.read(window=win);v=g[1]>0\\n     if v.mean()<.5:continue\\n     raw=S.read(list(range(1,14)),window=win).astype(np.float32);j=J.read(1,window=win);assert np.isin(j[v],[0,1,2,3]).all()\\n     x=(raw-NORM[:,0,None,None])/NORM[:,1,None,None]\\n     with torch.inference_mode():p=(m(torch.from_numpy(x)[None].cuda())[0,1]>0).cpu().numpy()\\n     w=g[1]==2;n+=1\\n     for code,kind in names.items():\\n      mask=v&(j==code);coverage[kind]['valid_pixels']+=int(mask.sum());coverage[kind]['gt_water_pixels']+=int((mask&w).sum())\\n      for region,reg in [('all',v),('clear',g[0]==1),('cloud',g[0]==2)]:counts[kind][region]+=cm(p,w,mask&reg)\\n  rows.append(dict(split=sp,scene=name,tiles=n,coverage=coverage,confusions={k:{g:a.tolist() for g,a in groups.items()} for k,groups in counts.items()}));print('PERMANENT FLOOD COMPLETION',sp,name,flush=True)\\ndef summarize(rs,kind,region):\\n mat=np.array([r['confusions'][kind][region] for r in rs]);vals=[iou(a) for a in mat if a[0]+a[2]>=200];rng=np.random.default_rng(42)\\n return dict(pooled_confusion=mat.sum(0).tolist(),pooled_iou_secondary=iou(mat.sum(0)),eligible_scenes=len(vals),scene_mean_iou=float(np.mean(vals)) if vals else None,ci95_scene_bootstrap=np.percentile([np.mean(rng.choice(vals,len(vals))) for _ in range(5000)],[2.5,97.5]).tolist() if vals else None)\\nsummary={sp:{k:{g:summarize([r for r in rows if sp=='val_test' or r['split']==sp],k,g) for g in ('all','clear','cloud')} for k in names.values()} for sp in ('val_test','val','test')}\\ncoverage={k:{field:sum(r['coverage'][k][field] for r in rows) for field in ('valid_pixels','gt_water_pixels')} for k in names.values()}\\nwater=sum(x['gt_water_pixels'] for x in coverage.values());unknown_fraction=coverage['unknown']['gt_water_pixels']/water\\nassert hashlib.sha256(BASE.read_bytes()).hexdigest()==before\\nout=dict(rule=rule,rows=rows,summary=summary,coverage=coverage,unknown_gt_water_fraction=unknown_fraction,full_genuine_flood_split='not identifiable from supplied JRC0 pixels; additional independent water-history/flood labels required',released_checkpoint_sha256=before,released_checkpoint_unchanged=True)\\n(R/'permanent_flood_completion.json').write_text(json.dumps(out,indent=2));print('JRC COMPLETION SUMMARY',json.dumps(dict(coverage=coverage,unknown_gt_water_fraction=unknown_fraction,summary=summary['val_test'])),flush=True)\\n\")\ncompile((audit/'permanent_flood_completion.py').read_text(),'permanent_flood_completion.py','exec')\nstream([sys.executable,'-u',str(audit/'permanent_flood_completion.py')])\nwith zipfile.ZipFile('/content/worldfloods-remaining-completion.zip','w',zipfile.ZIP_DEFLATED) as z:\n for name in ['permanent_flood_completion.py','PERMANENT_FLOOD_COMPLETION_RULE.md','results/permanent_flood_completion.json']:\n  p=audit/name;z.write(p,'clear-cloudy-gap-audit/'+name)\nprint('REMAINING FOLLOWUP ARCHIVE READY',flush=True)\nfiles.download('/content/worldfloods-remaining-completion.zip')\n")

PERMANENT FLOOD COMPLETION val EMSR260_07CAMPOGALLIANO_DEL_MONIT01_v1
PERMANENT FLOOD COMPLETION val EMSR271_02FARKADONA_DEL_v1
PERMANENT FLOOD COMPLETION val EMSR274_05TOAMASINA_DEL_v1
PERMANENT FLOOD COMPLETION val EMSR279_05ZARAGOZA_DEL_v2
PERMANENT FLOOD COMPLETION val EMSR279_11TORTOSA_DEL_v1
PERMANENT FLOOD COMPLETION val EMSR280_03FALUN_DEL_MONIT06_v2
PERMANENT FLOOD COMPLETION val EMSR293_03OZUN_GRA_v1
PERMANENT FLOOD COMPLETION val EMSR312_08CANDON_DEL_MONIT01_v1
PERMANENT FLOOD COMPLETION val EMSR329_05CASTIADAS_GRA_v3
PERMANENT FLOOD COMPLETION val EMSR358_AOI01_FEP_PRODUCT
PERMANENT FLOOD COMPLETION val EMSR427_AOI03_DEL_MONIT02
PERMANENT FLOOD COMPLETION val EMSR442_AOI03_DEL_PRODUCT
PERMANENT FLOOD COMPLETION val EMSR629_AOI01_DEL_PRODUCT
PERMANENT FLOOD COMPLETION val EMSR629_AOI02_DEL_PRODUCT
PERMANENT FLOOD COMPLETION val RS2_20161008_Water_Extent_Corail_Pestel
PERMANENT FLOOD COMPLETION val ST1_20161014_WaterExtent_BinhDinh_Lake
PERMANENT FLOOD COMPLETION test EMSR264

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import urllib.parse, hashlib
def hf_meta(url):
    with urllib.request.urlopen(url,timeout=60) as r:
        return json.load(r)
data_meta = hf_meta('https://huggingface.co/api/datasets/isp-uv-es/WorldFloodsv2?blobs=true')
model_meta = hf_meta('https://huggingface.co/api/models/isp-uv-es/ml4floods?blobs=true')
selected = [x for x in data_meta['siblings'] if x['rfilename'].split('/')[0] in ('val','test') and x['rfilename'].split('/')[1] in ('S2','gt','PERMANENTWATERJRC') and x['rfilename'].endswith('.tif')]
size = sum(x.get('size',0) for x in selected)
print('Required files',len(selected),'GiB',size/2**30,flush=True)
assert selected and size < shutil.disk_usage('/content').free*0.8
wf_root = pathlib.Path('/content/worldfloods')
wf_ckpt = pathlib.Path('/content/wf-models/model.pt')
os.environ['WF_ROOT'] = str(wf_root)
os.environ['WF_CKPT'] = str(wf_ckpt)
manifest = {'dataset_sha':data_meta['sha'],'checkpoint_sha':model_meta['sha'],'files':[]}
def download(base,revision,entry,target):
    target.parent.mkdir(parents=True,exist_ok=True)
    url = base+'/resolve/'+revision+'/'+urllib.parse.quote(entry['rfilename'],safe='/')
    with urllib.request.urlopen(url,timeout=120) as src, target.open('wb') as dst:
        shutil.copyfileobj(src,dst)
    assert target.stat().st_size == entry['size']
    h = hashlib.sha256()
    with target.open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''):
            h.update(chunk)
    if entry.get('lfs',{}).get('sha256'):
        assert h.hexdigest() == entry['lfs']['sha256']
    return {'path':str(target),'bytes':target.stat().st_size,'sha256':h.hexdigest()}
try:
    ck = next(x for x in model_meta['siblings'] if x['rfilename']=='models/WF2_unetv2_all/model.pt')
    manifest['files'].append(download('https://huggingface.co/isp-uv-es/ml4floods',model_meta['sha'],ck,wf_ckpt))
    print('CHECKPOINT DOWNLOADED',flush=True)
    for i,e in enumerate(selected):
        manifest['files'].append(download('https://huggingface.co/datasets/isp-uv-es/WorldFloodsv2',data_meta['sha'],e,wf_root/e['rfilename']))
        print('DOWNLOADED',i+1,len(selected),e['rfilename'],flush=True)
    manifest['complete'] = True
except Exception as error:
    manifest['complete'] = False
    manifest['error'] = str(error)
    print('STOP: DOWNLOAD FAILED',type(error).__name__,str(error),flush=True)
finally:
    (results/'download_manifest.json').write_text(json.dumps(manifest,indent=2))
print('DOWNLOAD COMPLETE',manifest['complete'])

Required files 102 GiB 5.235130960121751
CHECKPOINT DOWNLOADED
DOWNLOADED 1 102 test/PERMANENTWATERJRC/EMSR264_08VATOMANDRY_DEL_v2.tif
DOWNLOADED 2 102 test/PERMANENTWATERJRC/EMSR264_18MIANDRIVAZODETAIL_DEL_v2.tif
DOWNLOADED 3 102 test/PERMANENTWATERJRC/EMSR273_01GRILE_DEL_MONIT04_v2.tif
DOWNLOADED 4 102 test/PERMANENTWATERJRC/EMSR286_08ITUANGONORTH_DEL_MONIT02_v1.tif
DOWNLOADED 5 102 test/PERMANENTWATERJRC/EMSR286_09ITUANGOSOUTH_DEL_MONIT02_v1.tif
DOWNLOADED 6 102 test/PERMANENTWATERJRC/EMSR333_01RATTALORO_DEL_MONIT01_v1.tif
DOWNLOADED 7 102 test/PERMANENTWATERJRC/EMSR333_02PORTOPALO_DEL_MONIT01_v1.tif
DOWNLOADED 8 102 test/PERMANENTWATERJRC/EMSR342_06NORTHNORMANTON_DEL_v1.tif
DOWNLOADED 9 102 test/PERMANENTWATERJRC/EMSR342_07SOUTHNORMANTON_DEL_MONIT03_v2.tif
DOWNLOADED 10 102 test/PERMANENTWATERJRC/EMSR347_06MWANZA_DEL_v1.tif
DOWNLOADED 11 102 test/PERMANENTWATERJRC/EMSR347_07ZOMBA_DEL_MONIT01_v1.tif
DOWNLOADED 12 102 test/PERMANENTWATERJRC/EMSR347_07ZOMBA_DEL_v2.tif
DOWNLOADED 13 10

In [ ]:
exec("assert manifest['complete'], 'STOP: missing download'\nsubprocess.run([sys.executable,'-m','pip','install','-q','rasterio'],check=True)\nsys.path.insert(0,str(audit))\nimport numpy as np, rasterio\nfrom wf_data import scenes\nprint('SCENES', {sp:len(scenes(sp)) for sp in ('val','test')},flush=True)\nassert len(scenes('val'))==16 and len(scenes('test'))==18\nos.environ['OUT'] = 'results/cloud_iou_gpu'\nsource = (audit/'cloud_iou_pilot.py').read_text()\nsource = source.replace('m.eval()', \"m.eval(); m.to('cuda')\")\nsource = source.replace('m(torch.from_numpy(x)[None])', \"m(torch.from_numpy(x)[None].to('cuda'))\")\nsource = source.replace('[0].numpy()', '[0].cpu().numpy()')\nsaved_argv = sys.argv[:]\nsys.argv = ['cloud_iou_pilot.py','test','val']\ntry:\n    exec(compile(source,'cloud_iou_pilot_gpu','exec'),{'__name__':'__main__'})\nfinally:\n    sys.argv = saved_argv\nnew = json.loads((results/'cloud_iou_gpu_tiles.json').read_text())\nref = json.loads((audit/'cloud_iou_tiles.json').read_text())\nkey = lambda x:(x['split'],x['scene'],x['r'],x['c'])\na,b = {key(x):x for x in new},{key(x):x for x in ref}\ncomparison = {'tiles':len(a),'reference_tiles':len(b),'same_tile_keys':set(a)==set(b)}\nif comparison['same_tile_keys']:\n    changes = [sum(abs(a[k][v][i]-b[k][v][i]) for v in ('all','clear','under_cloud') for i in range(3)) for k in a]\n    comparison['tiles_with_count_differences'] = sum(x>0 for x in changes)\n    comparison['max_absolute_count_difference'] = max(changes)\n(results/'gpu_inference_comparison.json').write_text(json.dumps(comparison,indent=2))\nprint('INFERENCE COMPARISON',json.dumps(comparison),flush=True)")

SCENES {'val': 16, 'test': 18}
test EMSR264_08VATOMANDRY_DEL_v2 55
test EMSR264_18MIANDRIVAZODETAIL_DEL_v2 64
test EMSR273_01GRILE_DEL_MONIT04_v2 68
test EMSR286_08ITUANGONORTH_DEL_MONIT02_v1 94
test EMSR286_09ITUANGOSOUTH_DEL_MONIT02_v1 119
test EMSR333_01RATTALORO_DEL_MONIT01_v1 125
test EMSR333_02PORTOPALO_DEL_MONIT01_v1 133
test EMSR342_06NORTHNORMANTON_DEL_v1 234
test EMSR342_07SOUTHNORMANTON_DEL_MONIT03_v2 355
test EMSR347_06MWANZA_DEL_v1 535
test EMSR347_07ZOMBA_DEL_MONIT01_v1 576
test EMSR347_07ZOMBA_DEL_v2 580
test EMSR419_AOI01_DEL_MONIT01 594
test EMSR422_AOI06_DEL_PRODUCT 598
test EMSR438_AOI02_DEL_MONIT03 645
test EMSR438_AOI06_DEL_PRODUCT 649
test EMSR466_AOI01_DEL_PRODUCT 664
test EMSR9284_01YLITORNIONORTHERN_DEL_MONIT01_v1 676
val EMSR260_07CAMPOGALLIANO_DEL_MONIT01_v1 680
val EMSR271_02FARKADONA_DEL_v1 700
val EMSR274_05TOAMASINA_DEL_v1 725
val EMSR279_05ZARAGOZA_DEL_v2 812
val EMSR279_11TORTOSA_DEL_v1 928
val EMSR280_03FALUN_DEL_MONIT06_v2 982
val EMSR293_03OZUN_GRA_v

In [ ]:
exec("(audit/'pooled_placebo_followup.py').write_text(\"import numpy as np, rasterio\\nfrom rasterio.windows import Window\\nfrom wf_data import scenes, NORM\\nassert manifest['complete']\\nrule = '''Follow-up registered before new results. Frozen WF2_unetv2_all; 256px grid, stride 512, >=50% valid water labels; clear cf<0.01, donor cf>=0.1; eligible scenes >=200 clear-water pixels. Same-scene donors cycle row/col. Fallback donors are sorted within split with labelled-pixel cf in [0.1,0.6], index (i+42)%K. Corrected cloud fill: exact per-band pooled median over every GT-cloud pixel in kept scene tiles if >=1000 cloud pixels, else exact pooled validation-cloud median. Non-cloud placebo: exact per-band pooled median of valid GT-clear pixels of the same scene, or frozen pooled validation clear pixels if <1000. Identical masks for both fills; constant hard paste; score only unpainted valid pixels. This placebo measures generic constant-fill disruption, not natural cloud. Scene confusion pools, unweighted scene means, 5000 scene bootstrap draws, seed 42; original result unchanged. No tuning. Exact integer histograms compute medians without subsampling.'''\\n(audit/'FOLLOWUP_POOLED_PLACEBO_RULE.md').write_text(rule)\\nnamespace = {}\\nexec((audit/'cloud_iou_pilot.py').read_text().split('BINS=')[0], namespace)\\nmodel = namespace['m'].to('cuda').eval()\\n@torch.inference_mode()\\ndef predict(raw):\\n    x = (raw.astype(np.float32)-NORM[:,0,None,None])/NORM[:,1,None,None]\\n    return (torch.sigmoid(model(torch.from_numpy(x)[None].to('cuda')))[0,1]>0.5).cpu().numpy()\\ndef cm(p,w,v):\\n    return np.array([(p&w&v).sum(),(p&~w&v).sum(),(~p&w&v).sum()],np.int64)\\ndef iou(a):\\n    return float(a[0]/a.sum()) if a.sum() else None\\ndef median_hist(h):\\n    n=int(h[0].sum())\\n    assert n>0\\n    c=np.cumsum(h,axis=1)\\n    return np.array([(np.searchsorted(row,(n-1)//2+1)+np.searchsorted(row,n//2+1))/2 for row in c],np.float32)\\ndef add_hist(h,raw,mask):\\n    for band in range(13):\\n        vals=raw[band,mask]\\n        assert np.all(vals==vals.astype(np.uint16))\\n        h[band] += np.bincount(vals.astype(np.int64),minlength=65536)\\ninfo = {'val':[],'test':[]}\\nfor split in info:\\n    for name,s2,gt in scenes(split):\\n        ts=[]; hc=np.zeros((13,65536),np.int64); hn=hc.copy()\\n        with rasterio.open(s2) as S, rasterio.open(gt) as G:\\n            for r in range(0,G.height-255,512):\\n                for c in range(0,G.width-255,512):\\n                    win=Window(c,r,256,256); g=G.read(window=win); v=g[1]>0\\n                    if v.mean()<0.5: continue\\n                    raw=S.read(list(range(1,14)),window=win)\\n                    f=float((g[0]==2)[v].mean())\\n                    ts.append((r,c,g,f))\\n                    add_hist(hc,raw,g[0]==2)\\n                    add_hist(hn,raw,(g[0]==1)&v)\\n        info[split].append(dict(name=name,s2=s2,tiles=ts,hc=hc,hn=hn))\\n        print('EXACT SCAN',split,name,len(ts),flush=True)\\nfrozen_cloud=median_hist(sum(s['hc'] for s in info['val']))\\nfrozen_clear=median_hist(sum(s['hn'] for s in info['val']))\\nrows=[]\\nfor split,scs in info.items():\\n    donors=[(s['name'],r,c,g[0]==2) for s in scs for r,c,g,f in s['tiles'] if 0.1<=f<=0.6]\\n    for s in scs:\\n        clear=[(r,c,g) for r,c,g,f in s['tiles'] if f<0.01]\\n        cw=sum(int(((g[0]==1)&(g[1]==2)).sum()) for r,c,g in clear)\\n        if cw<200: continue\\n        own=[g[0]==2 for r,c,g,f in s['tiles'] if f>=0.1]\\n        assert own or donors\\n        cloud=median_hist(s['hc']) if s['hc'][0].sum()>=1000 else frozen_cloud\\n        noncloud=median_hist(s['hn']) if s['hn'][0].sum()>=1000 else frozen_clear\\n        acc={k:np.zeros(3,np.int64) for k in ('original','cloud','placebo')}\\n        with rasterio.open(s['s2']) as S:\\n            for i,(r,c,g) in enumerate(clear):\\n                mask=own[i%len(own)] if own else donors[(i+42)%len(donors)][3]\\n                raw=S.read(list(range(1,14)),window=Window(c,r,256,256)).astype(np.float32)\\n                valid=(g[1]>0)&~mask; water=g[1]==2\\n                acc['original']+=cm(predict(raw),water,valid)\\n                for kind,spectrum in [('cloud',cloud),('placebo',noncloud)]:\\n                    painted=raw.copy(); painted[:,mask]=spectrum[:,None]\\n                    acc[kind]+=cm(predict(painted),water,valid)\\n        scores={k:iou(v) for k,v in acc.items()}\\n        row=dict(split=split,scene=s['name'],n_clear_tiles=len(clear),clear_water_px=cw,own_mask=bool(own),own_cloud_spectrum=bool(s['hc'][0].sum()>=1000),cloud_spectrum=cloud.tolist(),noncloud_spectrum=noncloud.tolist(),confusions={k:v.tolist() for k,v in acc.items()},iou=scores)\\n        row['gap_cloud_pts']=100*(scores['original']-scores['cloud'])\\n        row['gap_placebo_pts']=100*(scores['original']-scores['placebo'])\\n        row['cloud_minus_placebo_pts']=row['gap_cloud_pts']-row['gap_placebo_pts']\\n        rows.append(row)\\n        (results/'pooled_placebo_progress.json').write_text(json.dumps(rows,indent=2))\\n        print('POOLED PLACEBO',split,s['name'],row['gap_cloud_pts'],row['gap_placebo_pts'],flush=True)\\ndef summary(rs,key):\\n    g=np.array([r[key] for r in rs],float)\\n    rng=np.random.default_rng(42)\\n    bs=np.array([rng.choice(g,len(g)).mean() for _ in range(5000)])\\n    return dict(scenes=len(g),mean_gap_pts=float(g.mean()),ci95=np.percentile(bs,[2.5,97.5]).tolist())\\nsummary_result={sp:{k:summary([r for r in rows if sp=='val_test' or r['split']==sp],k) for k in ('gap_cloud_pts','gap_placebo_pts','cloud_minus_placebo_pts')} for sp in ('val_test','val','test')}\\noutput=dict(rule=rule,seed=42,device='cuda',frozen_val_cloud_spectrum=frozen_cloud.tolist(),frozen_val_clear_spectrum=frozen_clear.tolist(),rows=rows,summary=summary_result)\\n(results/'pooled_cloud_noncloud_placebo.json').write_text(json.dumps(output,indent=2))\\nprint('PASTE SUMMARY',json.dumps(summary_result),flush=True)\\n\")\nexec(compile((audit/'pooled_placebo_followup.py').read_text(),'pooled_placebo_followup.py','exec'),globals())")

EXACT SCAN val EMSR260_07CAMPOGALLIANO_DEL_MONIT01_v1 4
EXACT SCAN val EMSR271_02FARKADONA_DEL_v1 20
EXACT SCAN val EMSR274_05TOAMASINA_DEL_v1 25
EXACT SCAN val EMSR279_05ZARAGOZA_DEL_v2 87
EXACT SCAN val EMSR279_11TORTOSA_DEL_v1 116
EXACT SCAN val EMSR280_03FALUN_DEL_MONIT06_v2 54
EXACT SCAN val EMSR293_03OZUN_GRA_v1 12
EXACT SCAN val EMSR312_08CANDON_DEL_MONIT01_v1 30
EXACT SCAN val EMSR329_05CASTIADAS_GRA_v3 4
EXACT SCAN val EMSR358_AOI01_FEP_PRODUCT 2
EXACT SCAN val EMSR427_AOI03_DEL_MONIT02 172
EXACT SCAN val EMSR442_AOI03_DEL_PRODUCT 1
EXACT SCAN val EMSR629_AOI01_DEL_PRODUCT 35
EXACT SCAN val EMSR629_AOI02_DEL_PRODUCT 35
EXACT SCAN val RS2_20161008_Water_Extent_Corail_Pestel 4
EXACT SCAN val ST1_20161014_WaterExtent_BinhDinh_Lake 14
EXACT SCAN test EMSR264_08VATOMANDRY_DEL_v2 55
EXACT SCAN test EMSR264_18MIANDRIVAZODETAIL_DEL_v2 9
EXACT SCAN test EMSR273_01GRILE_DEL_MONIT04_v2 4
EXACT SCAN test EMSR286_08ITUANGONORTH_DEL_MONIT02_v1 26
EXACT SCAN test EMSR286_09ITUANGOSOUTH_DEL_M

In [ ]:
exec("(audit/'jrc_brightness_followup.py').write_text(\"rule_strata = '''Registered before census/strata results. Same frozen WF2 baseline and 256px/512 grid. Brightness=sqrt(B2^2+B3^2+B4^2)/10000, per official ml4floods get_brightness at commit 192426c7286315b59a2f52c7a9cdebe589242de2; cuts .3 and .35; count GT-cloud pixels above/below each cut, with water counts, on valid water-label pixels. JRC strata: permanent JRC==3; nonpermanent JRC in {1,2}; unknown JRC==0 excluded from strata but counted. Compute binary water IoU within each spatial stratum, confusion pooled within scenes; report unweighted scene mean and 5000 scene-bootstrap CI seed42, plus pooled IoU explicitly as secondary. Nonpermanent water is a JRC-based proxy, not independently confirmed flood water. Also report raw separate per-scene IoU for Campo Galliano and Ituango South; within-scene tile bootstrap 5000 draws seed42 is descriptive and not an event interval. Never add these to existing gap means.'''\\n(audit/'JRC_BRIGHTNESS_SCENE_RULE.md').write_text(rule_strata)\\nstrata_rows=[]; census_rows=[]; separate=[]\\nfor split,scs in info.items():\\n    for s in scs:\\n        name=s['name']; jrc=wf_root/split/'PERMANENTWATERJRC'/(name+'.tif')\\n        assert jrc.exists(), 'STOP missing JRC raster'\\n        counts={k:np.zeros(3,np.int64) for k in ('permanent','nonpermanent','all')}\\n        buckets={str(cut):{k:dict(pixels=0,water_pixels=0) for k in ('gt_cloud_le_cut','gt_cloud_gt_cut','gt_clear_le_cut','gt_clear_gt_cut')} for cut in (0.3,0.35)}\\n        jrc_counts={}; tile_cm=[]\\n        with rasterio.open(s['s2']) as S, rasterio.open(jrc) as J:\\n            assert S.shape==J.shape and S.transform==J.transform and S.crs==J.crs, 'STOP: JRC alignment differs'\\n            for r,c,g,f in s['tiles']:\\n                win=Window(c,r,256,256)\\n                raw=S.read(list(range(1,14)),window=win).astype(np.float32)\\n                j=J.read(1,window=win); v=g[1]>0; wt=g[1]==2\\n                pred=predict(raw)\\n                counts['all']+=cm(pred,wt,v)\\n                counts['permanent']+=cm(pred,wt,v&(j==3))\\n                counts['nonpermanent']+=cm(pred,wt,v&np.isin(j,[1,2]))\\n                tile_cm.append(cm(pred,wt,v).tolist())\\n                for label,n in zip(*np.unique(j[v],return_counts=True)):\\n                    jrc_counts[str(int(label))]=jrc_counts.get(str(int(label)),0)+int(n)\\n                brightness=np.sqrt(np.sum(raw[[1,2,3]]**2,axis=0))/10000.\\n                for cut in (0.3,0.35):\\n                    for cloud_code,prefix in [(2,'gt_cloud'),(1,'gt_clear')]:\\n                        for side,bright in [('le_cut',brightness<=cut),('gt_cut',brightness>cut)]:\\n                            mask=v&(g[0]==cloud_code)&bright\\n                            b=buckets[str(cut)][prefix+'_'+side]\\n                            b['pixels']+=int(mask.sum()); b['water_pixels']+=int((mask&wt).sum())\\n        strata_rows.append(dict(split=split,scene=name,n_tiles=len(s['tiles']),jrc_pixel_counts=jrc_counts,confusions={k:v.tolist() for k,v in counts.items()},iou={k:iou(v) for k,v in counts.items()}))\\n        census_rows.append(dict(split=split,scene=name,cuts=buckets))\\n        if name.startswith('EMSR260_07CAMPOGALLIANO') or name.startswith('EMSR286_09ITUANGOSOUTH'):\\n            rng=np.random.default_rng(42); mat=np.array(tile_cm); values=[]\\n            for _ in range(5000):\\n                vv=iou(mat[rng.integers(0,len(mat),len(mat))].sum(0))\\n                if vv is not None: values.append(vv)\\n            separate.append(dict(split=split,scene=name,tiles=len(mat),iou=iou(counts['all']),ci95_tile_bootstrap=np.percentile(values,[2.5,97.5]).tolist() if values else None,uncertainty='within-scene tile bootstrap; descriptive; not event uncertainty',confusion=counts['all'].tolist()))\\n        print('JRC CENSUS',split,name,flush=True)\\ndef stratum_summary(rs,kind):\\n    vals=[r['iou'][kind] for r in rs if r['iou'][kind] is not None and sum(r['confusions'][kind][::2])>=200]\\n    rng=np.random.default_rng(42)\\n    ci=np.percentile([np.mean(rng.choice(vals,len(vals))) for _ in range(5000)],[2.5,97.5]).tolist() if vals else None\\n    pooled=np.array([r['confusions'][kind] for r in rs]).sum(0)\\n    return dict(scenes=len(vals),scene_gate='>=200 GT-water pixels in stratum',scene_mean_iou=float(np.mean(vals)) if vals else None,ci95_scene_bootstrap=ci,pooled_iou_secondary=iou(pooled),pooled_confusion=pooled.tolist())\\nstrata_summary={split:{k:stratum_summary([r for r in strata_rows if split=='val_test' or r['split']==split],k) for k in ('permanent','nonpermanent','all')} for split in ('val_test','val','test')}\\ncensus_total={str(cut):{k:{field:sum(r['cuts'][str(cut)][k][field] for r in census_rows) for field in ('pixels','water_pixels')} for k in ('gt_cloud_le_cut','gt_cloud_gt_cut','gt_clear_le_cut','gt_clear_gt_cut')} for cut in (.3,.35)}\\n(results/'permanent_nonpermanent_water.json').write_text(json.dumps(dict(rule=rule_strata,rows=strata_rows,summary=strata_summary),indent=2))\\n(results/'brightness_census.json').write_text(json.dumps(dict(rule=rule_strata,rows=census_rows,total=census_total),indent=2))\\n(results/'separate_scene_measurements.json').write_text(json.dumps(dict(rule=rule_strata,scenes=separate),indent=2))\\nprint('STRATA SUMMARY',json.dumps(strata_summary),flush=True)\\nprint('BRIGHTNESS CENSUS',json.dumps(census_total),flush=True)\\nprint('SEPARATE SCENES',json.dumps(separate),flush=True)\\n\")\nexec(compile((audit/'jrc_brightness_followup.py').read_text(),'jrc_brightness_followup.py','exec'),globals())")

JRC CENSUS val EMSR260_07CAMPOGALLIANO_DEL_MONIT01_v1
JRC CENSUS val EMSR271_02FARKADONA_DEL_v1
JRC CENSUS val EMSR274_05TOAMASINA_DEL_v1
JRC CENSUS val EMSR279_05ZARAGOZA_DEL_v2
JRC CENSUS val EMSR279_11TORTOSA_DEL_v1
JRC CENSUS val EMSR280_03FALUN_DEL_MONIT06_v2
JRC CENSUS val EMSR293_03OZUN_GRA_v1
JRC CENSUS val EMSR312_08CANDON_DEL_MONIT01_v1
JRC CENSUS val EMSR329_05CASTIADAS_GRA_v3
JRC CENSUS val EMSR358_AOI01_FEP_PRODUCT
JRC CENSUS val EMSR427_AOI03_DEL_MONIT02
JRC CENSUS val EMSR442_AOI03_DEL_PRODUCT
JRC CENSUS val EMSR629_AOI01_DEL_PRODUCT
JRC CENSUS val EMSR629_AOI02_DEL_PRODUCT
JRC CENSUS val RS2_20161008_Water_Extent_Corail_Pestel
JRC CENSUS val ST1_20161014_WaterExtent_BinhDinh_Lake
JRC CENSUS test EMSR264_08VATOMANDRY_DEL_v2
JRC CENSUS test EMSR264_18MIANDRIVAZODETAIL_DEL_v2
JRC CENSUS test EMSR273_01GRILE_DEL_MONIT04_v2
JRC CENSUS test EMSR286_08ITUANGONORTH_DEL_MONIT02_v1
JRC CENSUS test EMSR286_09ITUANGOSOUTH_DEL_MONIT02_v1
JRC CENSUS test EMSR333_01RATTALORO_DEL_MONIT

In [ ]:
stream([prithvi_python,'-c',"import os; os.environ['MPLBACKEND']='Agg'; import inspect; from terratorch.cli_tools import LightningInferenceModel; import terratorch.datamodules.sen1floods11 as s; print('PRITHVI HELPER',inspect.getsource(LightningInferenceModel.from_config)); print('SEN1 SOURCE',inspect.getsource(s))"])

INFO:albumentations.check_version:A new version of Albumentations is available: 2.0.8 (you have 1.4.10). Upgrade using: pip install --upgrade albumentations
INFO:matplotlib.font_manager:generated new fontManager
PRITHVI HELPER     @staticmethod
    def from_config(
        config_path: Path,
        checkpoint_path: Path | None = None,
        predict_dataset_bands: list[str] | None = None,
        predict_output_bands: list[str] | None = None,
    ):
        """
        Args:
            config_path (Path): Path to the config of the model to be loaded.
            checkpoint_path (Path): Path to the checkpoint to be loaded.
            predict_dataset_bands (list[str] | None, optional): List of bands present in input data.
                Defaults to None.
        """
        # use cli only to load
        arguments = [
            "--config",
            config_path,
        ]

        if predict_dataset_bands is not None:
            arguments.extend([ "--data.init_args.predict_data

In [ ]:
exec("(audit/'prithvi_smoke.py').write_text(\"import os\\nos.environ['MPLBACKEND']='Agg'\\nos.environ['NO_ALBUMENTATIONS_UPDATE']='1'\\nimport sys,json,pathlib,yaml,numpy as np,torch,rasterio\\nfrom rasterio.windows import Window\\nfrom terratorch.tasks import SemanticSegmentationTask\\nfrom terratorch.datamodules.sen1floods11 import MEANS,STDS\\naudit=pathlib.Path(__file__).resolve().parent\\nos.chdir(audit)\\nsys.path.insert(0,str(audit))\\nfrom wf_data import scenes\\nresults=audit/'results'; pdir=audit/'prithvi_assets'\\nconfig=yaml.safe_load((pdir/'config.yaml').read_text())\\nargs=config['model']['init_args']\\nargs['model_args']['backbone_pretrained']=False\\nprint('BUILDING PRITHVI',flush=True)\\nprithvi_task=SemanticSegmentationTask(**args)\\nweights=torch.load(pdir/'Prithvi-EO-V2-300M-TL-Sen1Floods11.pt',map_location='cpu',weights_only=True)\\nprint('CHECKPOINT KEYS',list(weights)[:8],flush=True)\\nif 'state_dict' in weights: weights=weights['state_dict']\\nif all(k.startswith('model.') for k in weights):\\n    prithvi_task.load_state_dict(weights,strict=True)\\nelse:\\n    prithvi_task.model.load_state_dict(weights,strict=True)\\ndel weights\\nprithvi_model=prithvi_task.model.to('cuda').eval()\\nbands=config['data']['init_args']['bands']\\nmeans=np.array([MEANS[b] for b in bands],np.float32)[:,None,None]\\nstds=np.array([STDS[b] for b in bands],np.float32)[:,None,None]\\n@torch.inference_mode()\\ndef predict(raw):\\n    image=(raw[[1,2,3,8,11,12]].astype(np.float32)/10000.-means)/stds\\n    output=prithvi_model(torch.from_numpy(image)[None].to('cuda'),temporal_coords=None,location_coords=None).output\\n    assert output.shape==(1,2,256,256), str(output.shape)\\n    return (output.argmax(1)[0]==1).cpu().numpy()\\n\\nname,s2,gt=scenes('val')[0]\\nwith rasterio.open(s2) as S:\\n    raw=S.read(list(range(1,14)),window=Window(0,0,256,256))\\npred=predict(raw)\\nprint('PRITHVI SMOKE PASSED',pred.shape,int(pred.sum()),flush=True)\\n\")\nstream([prithvi_python,str(audit/'prithvi_smoke.py')])")

BUILDING PRITHVI
CHECKPOINT KEYS ['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'hparams_name', 'hyper_parameters']
PRITHVI SMOKE PASSED (256, 256) 5580


In [ ]:
exec("(audit/'prithvi_evaluate.py').write_text(\"import os\\nos.environ['MPLBACKEND']='Agg'\\nos.environ['NO_ALBUMENTATIONS_UPDATE']='1'\\nimport sys,json,pathlib,yaml,numpy as np,torch,rasterio\\nfrom rasterio.windows import Window\\nfrom terratorch.tasks import SemanticSegmentationTask\\nfrom terratorch.datamodules.sen1floods11 import MEANS,STDS\\naudit=pathlib.Path(__file__).resolve().parent\\nos.chdir(audit)\\nsys.path.insert(0,str(audit))\\nfrom wf_data import scenes\\nresults=audit/'results'; pdir=audit/'prithvi_assets'\\nconfig=yaml.safe_load((pdir/'config.yaml').read_text())\\nargs=config['model']['init_args']\\nargs['model_args']['backbone_pretrained']=False\\nprint('BUILDING PRITHVI',flush=True)\\nprithvi_task=SemanticSegmentationTask(**args)\\nweights=torch.load(pdir/'Prithvi-EO-V2-300M-TL-Sen1Floods11.pt',map_location='cpu',weights_only=True)\\nprint('CHECKPOINT KEYS',list(weights)[:8],flush=True)\\nif 'state_dict' in weights: weights=weights['state_dict']\\nif all(k.startswith('model.') for k in weights):\\n    prithvi_task.load_state_dict(weights,strict=True)\\nelse:\\n    prithvi_task.model.load_state_dict(weights,strict=True)\\ndel weights\\nprithvi_model=prithvi_task.model.to('cuda').eval()\\nbands=config['data']['init_args']['bands']\\nmeans=np.array([MEANS[b] for b in bands],np.float32)[:,None,None]\\nstds=np.array([STDS[b] for b in bands],np.float32)[:,None,None]\\n@torch.inference_mode()\\ndef predict(raw):\\n    image=(raw[[1,2,3,8,11,12]].astype(np.float32)/10000.-means)/stds\\n    output=prithvi_model(torch.from_numpy(image)[None].to('cuda'),temporal_coords=None,location_coords=None).output\\n    assert output.shape==(1,2,256,256), str(output.shape)\\n    return (output.argmax(1)[0]==1).cpu().numpy()\\n\\nimport builtins,runpy\\nfrom unittest.mock import patch\\nrule='Frozen Prithvi Sen1Floods11 checkpoint; strict full checkpoint loading; no training. 256x256 tiles, stride512, >=50% labelled pixels, identical to WF2. S2 indices [1,2,3,8,11,12], /10000 reflectance and official Sen1Floods11 means/stds; native 256px inference; no temporal/location metadata (config use_metadata=false); argmax of 2 logits. Composition: unchanged water-fraction and NDVI greedy matching and gates from committed script. Same-tile: identical corrected cloud/non-cloud spectra and donor-mask assignments from FOLLOWUP_POOLED_PLACEBO_RULE.md; unpainted valid pixels only. Aggregation: pooled confusion within scenes; unweighted scene means and5000 scene-bootstrap draws, seed0 for natural controls, seed42 for paste. Independent cross-dataset model, not a claim of WorldFloods training equivalence.'\\n(audit/'PRITHVI_SAME_PROTOCOL_RULE.md').write_text(rule)\\ndef cm(p,w,v):\\n    return np.array([(p&w&v).sum(),(p&~w&v).sum(),(~p&w&v).sum()],np.int64)\\ndef iou(v):\\n    return float(v[0]/sum(v)) if sum(v) else None\\ninfo={'val':[],'test':[]}\\nfor split in info:\\n    for name,s2,gt in scenes(split):\\n        ts=[]\\n        with rasterio.open(gt) as G:\\n            for r in range(0,G.height-255,512):\\n                for c in range(0,G.width-255,512):\\n                    g=G.read(window=Window(c,r,256,256)); v=g[1]>0\\n                    if v.mean()<.5: continue\\n                    ts.append((r,c,g,float((g[0]==2)[v].mean())))\\n        info[split].append(dict(name=name,s2=s2,tiles=ts))\\n\\nreference=json.loads((audit/'method1_tiles.json').read_text())\\nrefmap={(x['split'],x['scene'],x['r'],x['c']):x for x in reference}\\nnatural=[]; terrain=[]; originals={}\\nfor split,scs in info.items():\\n    for s in scs:\\n        with rasterio.open(s['s2']) as S:\\n            for r,c,g,f in s['tiles']:\\n                raw=S.read(list(range(1,14)),window=Window(c,r,256,256))\\n                p=predict(raw); v=g[1]>0; w=g[1]==2; cloud=g[0]==2\\n                key=(split,s['name'],r,c); originals[key]=p\\n                base=cm(p,w,v).tolist()\\n                natural.append(dict(split=split,scene=s['name'],r=r,c=c,cloud_frac=f,all=base,clear=cm(p,w,v&~cloud).tolist(),under_cloud=cm(p,w,v&cloud).tolist()))\\n                item=dict(refmap[key]); item['cm']={'base':base}; terrain.append(item)\\n        print('PRITHVI NATURAL',split,s['name'],len(natural),flush=True)\\nassert set(originals)==set(refmap)\\n(results/'prithvi_cloud_iou_tiles.json').write_text(json.dumps(natural))\\n(results/'prithvi_method1_tiles.json').write_text(json.dumps(terrain))\\nreal_open=builtins.open\\nmaps={'cloud_iou_tiles.json':results/'prithvi_cloud_iou_tiles.json','method1_tiles.json':results/'prithvi_method1_tiles.json'}\\nfor script,outname in [('kill_test.py','kill_test_results.json'),('ndvi_comatch_analyze.py','ndvi_comatch_results.json')]:\\n    def redirected(file,mode='r',*args,**kwargs):\\n        if isinstance(file,(str,os.PathLike)):\\n            name=pathlib.Path(file).name\\n            if any(c in mode for c in 'wax+'):\\n                assert name==outname\\n                file=results/('prithvi_'+outname)\\n            elif name in maps: file=maps[name]\\n        return real_open(file,mode,*args,**kwargs)\\n    with patch('builtins.open',redirected):\\n        runpy.run_path(str(audit/script),run_name='__main__')\\n    print('PRITHVI COMPOSITION SAVED',script,flush=True)\\ncontrols=json.loads((results/'pooled_cloud_noncloud_placebo.json').read_text())\\ncontrolmap={(r['split'],r['scene']):r for r in controls['rows']}\\nrows=[]\\nfor split,scs in info.items():\\n    donors=[(s['name'],r,c,g[0]==2) for s in scs for r,c,g,f in s['tiles'] if .1<=f<=.6]\\n    for s in scs:\\n        if (split,s['name']) not in controlmap: continue\\n        control=controlmap[(split,s['name'])]\\n        own=[g[0]==2 for r,c,g,f in s['tiles'] if f>=.1]\\n        clear=[(r,c,g) for r,c,g,f in s['tiles'] if f<.01]\\n        acc={k:np.zeros(3,np.int64) for k in ('original','cloud','placebo')}\\n        with rasterio.open(s['s2']) as S:\\n            for i,(r,c,g) in enumerate(clear):\\n                mask=own[i%len(own)] if own else donors[(i+42)%len(donors)][3]\\n                raw=S.read(list(range(1,14)),window=Window(c,r,256,256)).astype(np.float32)\\n                valid=(g[1]>0)&~mask; water=g[1]==2\\n                acc['original']+=cm(originals[(split,s['name'],r,c)],water,valid)\\n                for kind,spec_key in [('cloud','cloud_spectrum'),('placebo','noncloud_spectrum')]:\\n                    painted=raw.copy(); painted[:,mask]=np.array(control[spec_key],np.float32)[:,None]\\n                    acc[kind]+=cm(predict(painted),water,valid)\\n        scores={k:iou(v) for k,v in acc.items()}\\n        row=dict(split=split,scene=s['name'],n_clear_tiles=len(clear),confusions={k:v.tolist() for k,v in acc.items()},iou=scores)\\n        assert all(v is not None for v in scores.values()), 'Undefined IoU'\\n        row['gap_cloud_pts']=100*(scores['original']-scores['cloud'])\\n        row['gap_placebo_pts']=100*(scores['original']-scores['placebo'])\\n        row['cloud_minus_placebo_pts']=row['gap_cloud_pts']-row['gap_placebo_pts']\\n        rows.append(row)\\n        (results/'prithvi_paste_progress.json').write_text(json.dumps(rows,indent=2))\\n        print('PRITHVI PASTE',split,s['name'],row['gap_cloud_pts'],row['gap_placebo_pts'],flush=True)\\ndef summary(rs,key):\\n    g=np.array([r[key] for r in rs]); rng=np.random.default_rng(42)\\n    bs=[rng.choice(g,len(g)).mean() for _ in range(5000)]\\n    return dict(scenes=len(g),mean_gap_pts=float(g.mean()),ci95=np.percentile(bs,[2.5,97.5]).tolist())\\nsummary_result={sp:{k:summary([r for r in rows if sp=='val_test' or r['split']==sp],k) for k in ('gap_cloud_pts','gap_placebo_pts','cloud_minus_placebo_pts')} for sp in ('val_test','val','test')}\\n(results/'prithvi_same_protocol.json').write_text(json.dumps(dict(rule=rule,model_revision=json.loads((results/'prithvi_download_manifest.json').read_text())['revision'],rows=rows,summary=summary_result),indent=2))\\nbase=json.loads((results/'cloud_iou_gpu_tiles.json').read_text())\\ndef pooled(rs): return iou(np.array([r['all'] for r in rs]).sum(0))\\ncomparison={sp:{group:{'wf2_iou':pooled([r for r in base if r['split']==sp and filt(r)]),'prithvi_iou':pooled([r for r in natural if r['split']==sp and filt(r)])} for group,filt in [('all',lambda r:True),('clear',lambda r:r['cloud_frac']<.01),('cloudy',lambda r:r['cloud_frac']>=.1)]} for sp in ('val','test')}\\n(results/'prithvi_wf2_pooled_comparison.json').write_text(json.dumps(dict(aggregator='pixel pooled; secondary to scene-level controls',comparison=comparison),indent=2))\\nprint('PRITHVI PASTE SUMMARY',json.dumps(summary_result),flush=True)\\nprint('PRITHVI POOLED COMPARISON',json.dumps(comparison),flush=True)\\nprint('PRITHVI COMPLETE',flush=True)\\n\")\nstream([prithvi_python,str(audit/'prithvi_evaluate.py')])")

BUILDING PRITHVI
CHECKPOINT KEYS ['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'hparams_name', 'hyper_parameters']
PRITHVI NATURAL val EMSR260_07CAMPOGALLIANO_DEL_MONIT01_v1 4
PRITHVI NATURAL val EMSR271_02FARKADONA_DEL_v1 24
PRITHVI NATURAL val EMSR274_05TOAMASINA_DEL_v1 49
PRITHVI NATURAL val EMSR279_05ZARAGOZA_DEL_v2 136
PRITHVI NATURAL val EMSR279_11TORTOSA_DEL_v1 252
PRITHVI NATURAL val EMSR280_03FALUN_DEL_MONIT06_v2 306
PRITHVI NATURAL val EMSR293_03OZUN_GRA_v1 318
PRITHVI NATURAL val EMSR312_08CANDON_DEL_MONIT01_v1 348
PRITHVI NATURAL val EMSR329_05CASTIADAS_GRA_v3 352
PRITHVI NATURAL val EMSR358_AOI01_FEP_PRODUCT 354
PRITHVI NATURAL val EMSR427_AOI03_DEL_MONIT02 526
PRITHVI NATURAL val EMSR442_AOI03_DEL_PRODUCT 527
PRITHVI NATURAL val EMSR629_AOI01_DEL_PRODUCT 562
PRITHVI NATURAL val EMSR629_AOI02_DEL_PRODUCT 597
PRITHVI NATURAL val RS2_20161008_Water_Extent_Corail_Pestel 601
PRITHVI NATURAL val ST1_20161014_WaterExtent_BinhDinh_Lake 615
PRITHVI N

In [ ]:
exec("import os,sys,json,pathlib,subprocess,shutil,urllib.request\nrepo=pathlib.Path('/content/Reproducing-research-')\naudit=repo/'clear-cloudy-gap-audit'; results=audit/'results'\nwf_root=pathlib.Path(os.environ['WF_ROOT']); wf_ckpt=pathlib.Path(os.environ['WF_CKPT'])\nassert wf_root.exists() and wf_ckpt.exists()\nwith urllib.request.urlopen('https://huggingface.co/api/datasets/isp-uv-es/WorldFloodsv2/revision/1f3faa2989e69930ac31d5c0a79fd224461123f8?blobs=true',timeout=120) as response:\n    data_meta=json.load(response)\ntrain_files=[x for x in data_meta['siblings'] if x['rfilename'].startswith('train/') and x['rfilename'].split('/')[1] in ('gt','S2') and x['rfilename'].endswith('.tif')]\nheldout_names={pathlib.Path(x['rfilename']).stem for x in data_meta['siblings'] if x['rfilename'].split('/')[0] in ('val','test') and '/gt/' in x['rfilename']}\nheldout_events={name.split('_')[0] for name in heldout_names if name.startswith('EMSR')}\ntrain_names={pathlib.Path(x['rfilename']).stem for x in train_files if '/gt/' in x['rfilename']}\neligible_names=sorted(n for n in train_names if n not in heldout_names and not (n.startswith('EMSR') and n.split('_')[0] in heldout_events))\nchosen_files=[x for x in train_files if pathlib.Path(x['rfilename']).stem in eligible_names]\npreflight=dict(dataset_revision=data_meta['sha'],all_training_scenes=len(train_names),eligible_training_scenes=len(eligible_names),excluded_eval_event_scenes=len(train_names)-len(eligible_names),training_download_gib=sum(x['size'] for x in chosen_files)/2**30,free_gib=shutil.disk_usage('/content').free/2**30,eligible_names=eligible_names)\n(results/'finetune_data_preflight.json').write_text(json.dumps(preflight,indent=2))\nprint('TRAINING PREFLIGHT',json.dumps({k:v for k,v in preflight.items() if k!='eligible_names'}),flush=True)\nprint('GPU STATUS',subprocess.run(['nvidia-smi','--query-gpu=name,memory.free','--format=csv,noheader'],capture_output=True,text=True,check=True).stdout,flush=True)\n")

TRAINING PREFLIGHT {"dataset_revision": "1f3faa2989e69930ac31d5c0a79fd224461123f8", "all_training_scenes": 475, "eligible_training_scenes": 388, "excluded_eval_event_scenes": 87, "training_download_gib": 65.81920205615461, "free_gib": 99.16999435424805}
GPU STATUS Tesla T4, 14536 MiB



In [ ]:
exec("import concurrent.futures,urllib.parse,hashlib,time\nrun_dir=results/'finetune_v1'\nassert not run_dir.exists(), 'Training run already exists; do not overwrite'\nrun_dir.mkdir()\nrule='Candidate WF2_finetune_v1; released baseline is immutable. Target >=0.90 pooled water IoU over all valid pixels on the same test tiles; this is an aim, not a promised result. Train only from pinned WorldFloodsv2 train split, excluding exact eval scene names and all train EMSR activations represented in val/test. No val/test labels or masks used for gradient updates. Fine-tune all UNet weights from the released checkpoint. Train grid256/stride256, >=50% valid labels; dihedral transforms and modest radiometric augmentation; masked water BCE+soft Dice and auxiliary cloud BCE. AdamW, encoder LR5e-6/decoderLR5e-5, cosine decay, batch8, seed42. Up to16 epochs,512 optimizer steps per epoch, validation after each; early stop after4 epochs without eligible improvement. Select highest validation pooled water IoU, provided clear/cloudy validation IoU each stays within1 point of the frozen baseline. Fixed threshold0.5; no test-based checkpoint or threshold selection. Once selection is locked, evaluate test once and run composition, same-tile cloud/non-cloud paste and named scene aggregation controls. Existing test data were inspected in the earlier audit; they are not a newly blind benchmark. All new JSON under results/finetune_v1; new weights under checkpoints/finetune_v1.'\n(audit/'FINETUNE_V1_RULE.md').write_text(rule)\n(run_dir/'rule.json').write_text(json.dumps(dict(rule=rule,seed=42,target_metric='test pooled water IoU on all labelled pixels',target=0.90,data_preflight=preflight),indent=2))\nrequired_bytes=sum(x['size'] for x in chosen_files)\nassert required_bytes < shutil.disk_usage('/content').free-8*2**30, 'STOP insufficient disk for training files'\ntraining_manifest=dict(dataset_revision=data_meta['sha'],complete=False,files=[],rule=rule)\ndef fetch_train(entry):\n    dest=wf_root/entry['rfilename']; dest.parent.mkdir(parents=True,exist_ok=True)\n    url='https://huggingface.co/datasets/isp-uv-es/WorldFloodsv2/resolve/'+data_meta['sha']+'/'+urllib.parse.quote(entry['rfilename'],safe='/')\n    if not dest.exists() or dest.stat().st_size!=entry['size']:\n        with urllib.request.urlopen(url,timeout=180) as src,dest.open('wb') as dst:\n            shutil.copyfileobj(src,dst)\n    assert dest.stat().st_size==entry['size'], 'Incomplete training file: '+str(dest)\n    digest=hashlib.sha256()\n    with dest.open('rb') as f:\n        for chunk in iter(lambda:f.read(8388608),b''): digest.update(chunk)\n    expected=entry.get('lfs',{}).get('sha256')\n    assert not expected or digest.hexdigest()==expected, 'Checksum mismatch: '+str(dest)\n    return dict(path=str(dest),bytes=dest.stat().st_size,sha256=digest.hexdigest())\ntry:\n    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:\n        done=0\n        for start in range(0,len(chosen_files),4):\n            for item in pool.map(fetch_train,chosen_files[start:start+4]):\n                training_manifest['files'].append(item); done+=1\n                if done%20==0 or done==len(chosen_files):\n                    print('TRAIN DOWNLOAD',done,len(chosen_files),sum(x['bytes'] for x in training_manifest['files'])/2**30,'GiB',flush=True)\n                    (run_dir/'training_download_manifest.json').write_text(json.dumps(training_manifest,indent=2))\n    training_manifest['complete']=True\nexcept Exception as error:\n    training_manifest['error']=repr(error)\n    print('STOP: TRAINING DOWNLOAD FAILED',repr(error),flush=True)\nfinally:\n    (run_dir/'training_download_manifest.json').write_text(json.dumps(training_manifest,indent=2))\nprint('TRAIN DOWNLOAD COMPLETE',training_manifest['complete'],flush=True)\nassert training_manifest['complete'], 'STOP: required training download failed'\n")

TRAIN DOWNLOAD 20 776 3.4247457766905427 GiB
TRAIN DOWNLOAD 40 776 4.7322687758132815 GiB
TRAIN DOWNLOAD 60 776 5.3600810915231705 GiB
TRAIN DOWNLOAD 80 776 7.686780772171915 GiB
TRAIN DOWNLOAD 100 776 9.50167301017791 GiB
TRAIN DOWNLOAD 120 776 10.8577651809901 GiB
TRAIN DOWNLOAD 140 776 12.492140110582113 GiB
TRAIN DOWNLOAD 160 776 13.888506406918168 GiB
TRAIN DOWNLOAD 180 776 19.38934168126434 GiB
TRAIN DOWNLOAD 200 776 21.077041566371918 GiB
TRAIN DOWNLOAD 220 776 26.31715419422835 GiB
TRAIN DOWNLOAD 240 776 31.99698860105127 GiB
TRAIN DOWNLOAD 260 776 39.189430095255375 GiB
TRAIN DOWNLOAD 280 776 49.36379845254123 GiB
TRAIN DOWNLOAD 300 776 50.68918568547815 GiB
TRAIN DOWNLOAD 320 776 55.01102321408689 GiB
TRAIN DOWNLOAD 340 776 58.02681580372155 GiB
TRAIN DOWNLOAD 360 776 60.70701196976006 GiB
TRAIN DOWNLOAD 380 776 62.72013235092163 GiB
TRAIN DOWNLOAD 400 776 65.68415238987654 GiB
TRAIN DOWNLOAD 420 776 65.68839564081281 GiB
TRAIN DOWNLOAD 440 776 65.69326375424862 GiB
TRAIN DOW

In [ ]:
exec("(audit/'FINETUNE_V1_CALIBRATION_RULE.md').write_text(\"Predeclared before fine-tuning results: compare the released checkpoint, best eligible fine-tuned checkpoint, and best unconstrained validation checkpoint (deduplicate identical epochs). For each, test water-probability thresholds0.20..0.90 in0.05 increments on validation only. Select highest pooled validation water IoU subject to clear/cloudy validation each >=released baseline minus1 point; ties prefer threshold closest0.5, then higher scene-mean validation IoU. Encode the chosen threshold as a shift of conv_last.bias[1] by minus log(t/(1-t)), so the exported model retains the fixed0.5 prediction rule. This is an explicitly calibrated candidate, separate from the fixed-threshold fine-tune. Freeze the final selection before one new test evaluation; do not tune from test results. Report all curves, source weights, composition and same-tile controls, scene aggregation and uncertainty. The 90% goal refers to pooled all-valid-pixel water IoU and may not be achieved.\")\n(run_dir/'calibration_rule.json').write_text(json.dumps(dict(rule=\"Predeclared before fine-tuning results: compare the released checkpoint, best eligible fine-tuned checkpoint, and best unconstrained validation checkpoint (deduplicate identical epochs). For each, test water-probability thresholds0.20..0.90 in0.05 increments on validation only. Select highest pooled validation water IoU subject to clear/cloudy validation each >=released baseline minus1 point; ties prefer threshold closest0.5, then higher scene-mean validation IoU. Encode the chosen threshold as a shift of conv_last.bias[1] by minus log(t/(1-t)), so the exported model retains the fixed0.5 prediction rule. This is an explicitly calibrated candidate, separate from the fixed-threshold fine-tune. Freeze the final selection before one new test evaluation; do not tune from test results. Report all curves, source weights, composition and same-tile controls, scene aggregation and uncertainty. The 90% goal refers to pooled all-valid-pixel water IoU and may not be achieved.\",test_opened=False),indent=2))\n(audit/'train_candidate_v1.py').write_text(\"\\\"\\\"\\\"Fine-tune a separate WF2 candidate; never modify the released checkpoint.\\\"\\\"\\\"\\nimport os,sys,json,pathlib,time,math,random,hashlib,collections\\nos.environ.setdefault('GDAL_CACHEMAX','64')\\nimport numpy as np,torch,torch.nn.functional as F,rasterio\\nfrom rasterio.windows import Window\\nfrom torch.utils.data import Dataset,DataLoader,WeightedRandomSampler\\nAUDIT=pathlib.Path(__file__).resolve().parent\\nos.chdir(AUDIT); sys.path.insert(0,str(AUDIT))\\nfrom wf_data import NORM\\nRUN=AUDIT/'results'/'finetune_v1'; CKPTS=AUDIT/'checkpoints'/'finetune_v1'\\nCKPTS.mkdir(parents=True,exist_ok=True)\\nROOT=pathlib.Path(os.environ['WF_ROOT']); BASE=pathlib.Path(os.environ['WF_CKPT'])\\nassert json.loads((RUN/'training_download_manifest.json').read_text())['complete']\\nassert not (RUN/'selection.json').exists(), 'Completed run exists; do not overwrite'\\nSEED=42; EPOCHS=16; STEPS=512; BATCH=8\\nrandom.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)\\ntorch.set_num_threads(3)\\nconfig=dict(seed=SEED,epochs=EPOCHS,steps_per_epoch=STEPS,batch_size=BATCH,encoder_lr=5e-6,decoder_lr=5e-5,weight_decay=1e-4,water_bce_weight=.5,water_dice_weight=.5,cloud_bce_weight=.1,patience=4,min_delta=.0001,water_threshold=.5,selection='maximum eligible validation pooled water IoU; clear and cloudy each >=baseline minus1 point',target_test_iou=.9,train_only=True,training_stride=256,evaluation_stride=512,precision='AMP fp16 training; fp32 validation',augmentation='random right-angle rotations/flips; gain0.9..1.1 and per-band Gaussian offset std20DN',sampler='inverse scene tile-count times (1+2*cloudy+mixed-water); replacement',training_events='exclude evaluation EMSR activations')\\n(RUN/'config.json').write_text(json.dumps(config,indent=2))\\ndef digest(path):\\n    h=hashlib.sha256()\\n    with open(path,'rb') as f:\\n        for c in iter(lambda:f.read(8388608),b''): h.update(c)\\n    return h.hexdigest()\\nbase_digest=digest(BASE)\\nmanifest=json.loads((RUN/'rule.json').read_text())['data_preflight']\\nindex=[]\\nfor scene in manifest['eligible_names']:\\n    s2=ROOT/'train'/'S2'/(scene+'.tif'); gt=ROOT/'train'/'gt'/(scene+'.tif')\\n    assert s2.exists() and gt.exists(), 'STOP missing training path'\\n    with rasterio.open(s2) as S,rasterio.open(gt) as G:\\n        assert S.count>=13 and G.count==2 and S.shape==G.shape and S.transform==G.transform\\n        for r in range(0,G.height-255,256):\\n            for c in range(0,G.width-255,256):\\n                g=G.read(window=Window(c,r,256,256)); v=g[1]>0\\n                if v.mean()<.5: continue\\n                index.append(dict(split='train',scene=scene,r=r,c=c,cloud_frac=float((g[0]==2)[v].mean()),water_frac=float((g[1]==2)[v].mean())))\\n    if len(index)%500<20: print('TRAIN INDEX',scene,len(index),flush=True)\\nassert index, 'No eligible training tiles'\\n(RUN/'train_index.json').write_text(json.dumps(index))\\nval_index=[r for r in json.loads((AUDIT/'method1_tiles.json').read_text()) if r['split']=='val']\\nassert len(val_index)==615\\nassert not ({r['scene'] for r in index}&{r['scene'] for r in val_index})\\nprint('DATA READY',len(index),'train tiles',len(val_index),'validation tiles',flush=True)\\nclass Tiles(Dataset):\\n    def __init__(self,meta,augment=False): self.meta=meta; self.augment=augment; self.cache=collections.OrderedDict()\\n    def __len__(self): return len(self.meta)\\n    def handles(self,row):\\n        key=(row['split'],row['scene'])\\n        if key not in self.cache:\\n            self.cache[key]=(rasterio.open(ROOT/key[0]/'S2'/(key[1]+'.tif')),rasterio.open(ROOT/key[0]/'gt'/(key[1]+'.tif')))\\n            if len(self.cache)>8:\\n                _,old=self.cache.popitem(last=False)\\n                for f in old: f.close()\\n        self.cache.move_to_end(key); return self.cache[key]\\n    def __getitem__(self,i):\\n        row=self.meta[i]; S,G=self.handles(row); win=Window(row['c'],row['r'],256,256)\\n        raw=S.read(list(range(1,14)),window=win).astype(np.float32); g=G.read(window=win).astype(np.int64)\\n        if self.augment:\\n            k=np.random.randint(4); raw=np.rot90(raw,k,axes=(1,2)); g=np.rot90(g,k,axes=(1,2))\\n            if np.random.rand()<.5: raw=raw[:,:,::-1]; g=g[:,:,::-1]\\n            if np.random.rand()<.5: raw=raw[:,::-1,:]; g=g[:,::-1,:]\\n            gain=np.random.uniform(.9,1.1); offset=np.random.normal(0,20,(13,1,1)).astype(np.float32)\\n            raw=np.maximum(raw*gain+offset,0)\\n        x=np.ascontiguousarray((raw-NORM[:,0,None,None])/NORM[:,1,None,None])\\n        return torch.from_numpy(x),torch.from_numpy(np.ascontiguousarray(g)),i\\n    def close(self):\\n        for pair in self.cache.values():\\n            for f in pair: f.close()\\n        self.cache.clear()\\ndef worker_seed(worker_id):\\n    seed=torch.initial_seed()%2**32; np.random.seed(seed); random.seed(seed)\\ndef confusion(p,w,v): return np.array([(p&w&v).sum(),(p&~w&v).sum(),(~p&w&v).sum()],np.int64)\\ndef pooled(rows,which):\\n    selected=[r['all'] for r in rows if which=='all' or (which=='clear' and r['cloud_frac']<.01) or (which=='cloudy' and r['cloud_frac']>=.1)]\\n    a=np.array(selected).sum(0) if selected else np.zeros(3)\\n    return float(a[0]/a.sum()) if a.sum() else None\\nval_dataset=Tiles(val_index)\\nval_loader=DataLoader(val_dataset,batch_size=4,shuffle=False,num_workers=2,pin_memory=True,worker_init_fn=worker_seed)\\nns={}; exec((AUDIT/'cloud_iou_pilot.py').read_text().split('BINS=')[0],ns)\\nmodel=ns['m'].to('cuda'); initial={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}\\n@torch.inference_mode()\\ndef evaluate():\\n    model.eval(); rows=[]\\n    for x,g,ids in val_loader:\\n        pred=(model(x.to('cuda',non_blocking=True))[:,1]>0).cpu().numpy(); gs=g.numpy()\\n        for j,idx in enumerate(ids.tolist()):\\n            t=val_index[idx]; v=gs[j,1]>0; w=gs[j,1]==2; cloud=gs[j,0]==2\\n            rows.append(dict(split='val',scene=t['scene'],r=t['r'],c=t['c'],cloud_frac=t['cloud_frac'],all=confusion(pred[j],w,v).tolist(),clear=confusion(pred[j],w,v&~cloud).tolist(),under_cloud=confusion(pred[j],w,v&cloud).tolist()))\\n    return {k:pooled(rows,k) for k in ('all','clear','cloudy')},rows\\nbaseline,baseline_rows=evaluate()\\n(RUN/'baseline_validation.json').write_text(json.dumps(dict(metrics=baseline,rows=baseline_rows),indent=2))\\nprint('BASELINE VALIDATION',json.dumps(baseline),flush=True)\\nencoder=[]; decoder=[]\\nfor name,p in model.named_parameters(): (encoder if name.startswith('dconv_down') else decoder).append(p)\\nopt=torch.optim.AdamW([dict(params=encoder,lr=config['encoder_lr']),dict(params=decoder,lr=config['decoder_lr'])],weight_decay=config['weight_decay'])\\ndef lr_scale(step):\\n    if step<128: return max(.1,(step+1)/128)\\n    progress=(step-128)/max(EPOCHS*STEPS-128,1)\\n    return .1+.9*.5*(1+math.cos(math.pi*progress))\\nschedule=torch.optim.lr_scheduler.LambdaLR(opt,lr_scale)\\nscaler=torch.amp.GradScaler('cuda',init_scale=1024)\\nscene_counts=collections.Counter(r['scene'] for r in index)\\nweights=torch.tensor([(1+2*(r['cloud_frac']>=.1)+(0.01<r['water_frac']<.8))/scene_counts[r['scene']] for r in index],dtype=torch.double)\\ntrain_dataset=Tiles(index,augment=True)\\nbest_eligible=baseline['all']; best_any=-1.; selected=None; best_any_epoch=None; stale=0; history=[]\\ndef save_weights(path): torch.save({'network.'+k:v.detach().cpu() for k,v in model.state_dict().items()},path)\\nstart=time.time()\\nfor epoch in range(EPOCHS):\\n    generator=torch.Generator().manual_seed(SEED+epoch)\\n    sampler=WeightedRandomSampler(weights,num_samples=STEPS*BATCH,replacement=True,generator=generator)\\n    loader=DataLoader(train_dataset,batch_size=BATCH,sampler=sampler,num_workers=2,pin_memory=True,drop_last=True,worker_init_fn=worker_seed,generator=generator)\\n    model.train(); losses=[]; epoch_start=time.time()\\n    for step,(x,g,_) in enumerate(loader):\\n        x=x.to('cuda',non_blocking=True); g=g.to('cuda',non_blocking=True)\\n        valid=(g[:,1]>0).float(); target=(g[:,1]==2).float(); cv=(g[:,0]>0).float(); ct=(g[:,0]==2).float()\\n        opt.zero_grad(set_to_none=True)\\n        with torch.autocast('cuda',dtype=torch.float16): logits=model(x)\\n        water=logits[:,1].float(); cloud_logits=logits[:,0].float()\\n        bce=(F.binary_cross_entropy_with_logits(water,target,reduction='none')*valid).sum()/valid.sum().clamp_min(1)\\n        probs=water.sigmoid()*valid\\n        dice=1-(2*(probs*target).sum()+1)/(probs.sum()+(target*valid).sum()+1)\\n        cloud_loss=(F.binary_cross_entropy_with_logits(cloud_logits,ct,reduction='none')*cv).sum()/cv.sum().clamp_min(1)\\n        loss=.5*bce+.5*dice+.1*cloud_loss\\n        assert torch.isfinite(loss), 'Non-finite training loss'\\n        scaler.scale(loss).backward(); scaler.unscale_(opt)\\n        torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\\n        scaler.step(opt); scaler.update(); schedule.step(); losses.append(float(loss.detach()))\\n        if (step+1)%64==0: print('TRAIN STEP',epoch+1,step+1,'loss',float(np.mean(losses)),'seconds',round(time.time()-epoch_start,1),flush=True)\\n    metrics,rows=evaluate()\\n    eligible=metrics['clear']>=baseline['clear']-.01 and metrics['cloudy']>=baseline['cloudy']-.01\\n    change=max(float((model.state_dict()[k].detach().cpu()-v).abs().max()) for k,v in initial.items())\\n    record=dict(epoch=epoch+1,training_loss=float(np.mean(losses)),validation=metrics,eligible=eligible,max_weight_change=change,seconds=time.time()-epoch_start)\\n    history.append(record)\\n    assert change>0, 'Weights did not update'\\n    if metrics['all']>best_any:\\n        best_any=metrics['all']; best_any_epoch=epoch+1\\n        save_weights(CKPTS/'best_validation_model.pt')\\n    if eligible and metrics['all']>best_eligible+config['min_delta']:\\n        best_eligible=metrics['all']; selected=dict(epoch=epoch+1,path=str(CKPTS/'best_eligible_model.pt'),validation=metrics)\\n        save_weights(CKPTS/'best_eligible_model.pt'); stale=0\\n        (RUN/'selected_validation_tiles.json').write_text(json.dumps(rows))\\n    else: stale+=1\\n    torch.save(dict(model=model.state_dict(),optimizer=opt.state_dict(),scheduler=schedule.state_dict(),scaler=scaler.state_dict(),epoch=epoch+1,history=history,config=config),CKPTS/'last_training_state.pt')\\n    (RUN/'history.json').write_text(json.dumps(dict(config=config,baseline=baseline,epochs=history),indent=2))\\n    print('EPOCH RESULT',json.dumps(record),'BEST ELIGIBLE',best_eligible,flush=True)\\n    if stale>=config['patience']: print('EARLY STOP',flush=True); break\\nassert digest(BASE)==base_digest, 'Released checkpoint changed'\\nselection=dict(run='finetune_v1',selected=selected,baseline_validation=baseline,best_any_validation_iou=best_any,best_any_epoch=best_any_epoch,best_any_weights=str(CKPTS/'best_validation_model.pt'),test_opened=False,epochs_completed=len(history),released_checkpoint_sha256=base_digest,released_checkpoint_unchanged=True,training_seconds=time.time()-start,target=.90)\\n(RUN/'selection.json').write_text(json.dumps(selection,indent=2))\\nprint('TRAINING COMPLETE',json.dumps(selection),flush=True)\\n\")\ncompile((audit/'train_candidate_v1.py').read_text(),'train_candidate_v1.py','exec')\nprint('TRAINER VALIDATED; STARTING GPU FINE-TUNING',flush=True)\nwith (run_dir/'train_stdout.log').open('w') as log:\n process=subprocess.Popen([sys.executable,'-u',str(audit/'train_candidate_v1.py')],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)\n try:\n  for line in process.stdout:\n   log.write(line); log.flush(); print(line,end='',flush=True)\n  code=process.wait()\n  assert code==0, f'Trainer failed with exit code {code}'\n finally:\n  if process.poll() is None: process.kill()\n")

TRAINER VALIDATED; STARTING GPU FINE-TUNING
TRAIN INDEX EMSR258_01DIVJAKE_DEL_v2 4015
TRAIN INDEX EMSR265_07DAMMARIELESLYS_DEL_MONIT01_v1 4515
TRAIN INDEX EMSR265_24HONFLEUR_DEL_MONIT01_v1 5003
TRAIN INDEX EMSR339_03BEIRANW_DEL_v1 10004
TRAIN INDEX EMSR339_04BEIRANE_DEL_v1 10013
TRAIN INDEX EMSR429_AOI08_DEL_PRODUCT 17000
TRAIN INDEX EMSR429_AOI09_DEL_MONIT01 17001
TRAIN INDEX EMSR429_AOI09_DEL_PRODUCT 17002
TRAIN INDEX EMSR444_AOI03_DEL_PRODUCT 17518
TRAIN INDEX EMSR445_AOI02_DEL_PRODUCT 18018
TRAIN INDEX EMSR446_AOI03_DEL_PRODUCT 19019
TRAIN INDEX EMSR468_AOI01_FEP_PRODUCT 20001
TRAIN INDEX EMSR497_AOI01_DEL_PRODUCT 27513
TRAIN INDEX EMSR504_AOI07_GRA_PRODUCT 29014
TRAIN INDEX EMSR517_AOI01_DEL_PRODUCT 30009
TRAIN INDEX EMSR565_AOI06_DEL_MONIT01 32512
TRAIN INDEX EMSR570_AOI01_DEL_MONIT02 33512
DATA READY 38596 train tiles 615 validation tiles
BASELINE VALIDATION {"all": 0.8651451103384014, "clear": 0.8671038269617226, "cloudy": 0.6746440927636234}
TRAIN STEP 1 64 loss 0.241844556410

In [ ]:
exec("(audit/'calibrate_candidate_v1.py').write_text(\"\\\"\\\"\\\"Validation-only selection and bias calibration of separate model weights.\\\"\\\"\\\"\\nimport os,sys,json,pathlib,math,hashlib,collections\\nimport numpy as np,torch,rasterio\\nfrom rasterio.windows import Window\\nA=pathlib.Path(__file__).resolve().parent\\nos.chdir(A);sys.path.insert(0,str(A))\\nfrom wf_data import NORM\\nR=A/'results'/'finetune_v1'; C=A/'checkpoints'/'finetune_v1'\\nassert (R/'calibration_rule.json').exists()\\nassert not (R/'final_selection.json').exists(), 'Selection already frozen'\\nselection=json.loads((R/'selection.json').read_text())\\nBASE=pathlib.Path(os.environ['WF_CKPT']);ROOT=pathlib.Path(os.environ['WF_ROOT'])\\nmeta=[r for r in json.loads((A/'method1_tiles.json').read_text()) if r['split']=='val']\\nns={};exec((A/'cloud_iou_pilot.py').read_text().split('BINS=')[0],ns)\\nm=ns['m'].cuda().eval();torch.set_num_threads(3)\\ndef sha(p):return hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()\\ndef iou(x):return float(x[0]/sum(x)) if sum(x) else None\\ndef cm(p,w,v):return np.array([(p&w&v).sum(),(p&~w&v).sum(),(~p&w&v).sum()],np.int64)\\nthresholds=[round(.2+.05*i,2) for i in range(15)]\\nsources=[dict(name='released',path=str(BASE),epoch=0)]\\nif selection['selected']:sources.append(dict(name='best_eligible',**selection['selected']))\\nanypath=C/'best_validation_model.pt'\\nanyepoch=selection['best_any_epoch']\\nif anypath.exists() and not any(s['epoch']==anyepoch for s in sources):sources.append(dict(name='best_validation',path=str(anypath),epoch=anyepoch))\\nbaseline=json.loads((R/'baseline_validation.json').read_text())['metrics']\\nrows=[];best=None\\nfor source in sources:\\n weights=torch.load(source['path'],map_location='cpu',weights_only=True)\\n if 'state_dict' in weights:weights=weights['state_dict']\\n weights={k.removeprefix('network.'):v for k,v in weights.items()}\\n m.load_state_dict(weights,strict=True);del weights\\n counts={t:{g:np.zeros(3,np.int64) for g in ('all','clear','cloudy')} for t in thresholds}\\n scene_counts={t:collections.defaultdict(lambda:np.zeros(3,np.int64)) for t in thresholds}\\n for scene in sorted({r['scene'] for r in meta}):\\n  with rasterio.open(ROOT/'val'/'S2'/(scene+'.tif')) as S,rasterio.open(ROOT/'val'/'gt'/(scene+'.tif')) as G:\\n   for r in [r for r in meta if r['scene']==scene]:\\n    win=Window(r['c'],r['r'],256,256);raw=S.read(list(range(1,14)),window=win).astype(np.float32);g=G.read(window=win)\\n    x=(raw-NORM[:,0,None,None])/NORM[:,1,None,None]\\n    with torch.inference_mode():logits=m(torch.from_numpy(x)[None].cuda())[0,1].cpu().numpy()\\n    w=g[1]==2;v=g[1]>0\\n    group='clear' if r['cloud_frac']<.01 else 'cloudy' if r['cloud_frac']>=.1 else None\\n    for t in thresholds:\\n     a=cm(logits>math.log(t/(1-t)),w,v);counts[t]['all']+=a;scene_counts[t][scene]+=a\\n     if group:counts[t][group]+=a\\n  print('CALIBRATION VAL',source['name'],scene,flush=True)\\n for t in thresholds:\\n  metrics={g:iou(a) for g,a in counts[t].items()}\\n  eligible=metrics['clear']>=baseline['clear']-.01 and metrics['cloudy']>=baseline['cloudy']-.01\\n  scene_mean=float(np.mean([iou(a) for a in scene_counts[t].values() if iou(a) is not None]))\\n  row=dict(source=source,threshold=t,validation=metrics,eligible=eligible,scene_mean_iou=scene_mean,confusions={g:a.tolist() for g,a in counts[t].items()})\\n  rows.append(row)\\n  rank=(metrics['all'],-abs(t-.5),scene_mean)\\n  if eligible and (best is None or rank>best[0]):best=(rank,row)\\n (R/'calibration_curve.json').write_text(json.dumps(dict(baseline=baseline,rows=rows),indent=2))\\nassert best is not None\\nchosen=best[1];src=chosen['source'];t=chosen['threshold']\\nweights=torch.load(src['path'],map_location='cpu',weights_only=True)\\nif 'state_dict' in weights:weights=weights['state_dict']\\nweights={k.removeprefix('network.'):v.clone() for k,v in weights.items()}\\noriginal_bias=float(weights['conv_last.bias'][1]);shift=-math.log(t/(1-t))\\nweights['conv_last.bias'][1]+=shift\\npath=C/'calibrated_selected_model.pt'\\ntorch.save({'network.'+k:v for k,v in weights.items()},path)\\nfinal=dict(chosen=chosen,checkpoint=str(path),checkpoint_sha256=sha(path),source_sha256=sha(src['path']),water_bias_before=original_bias,water_bias_shift=shift,water_bias_after=float(weights['conv_last.bias'][1]),prediction_threshold=.5,test_opened=False,target_test_iou=.9)\\n(R/'final_selection.json').write_text(json.dumps(final,indent=2))\\nprint('FINAL VALIDATION SELECTION FROZEN',json.dumps(final),flush=True)\\n\")\n(audit/'evaluate_candidate_v1.py').write_text(\"\\\"\\\"\\\"Evaluate the already frozen candidate once under the existing controls.\\\"\\\"\\\"\\nimport os,sys,json,pathlib,numpy as np,torch,rasterio,hashlib\\nfrom rasterio.windows import Window\\nA=pathlib.Path(__file__).resolve().parent\\nos.chdir(A);sys.path.insert(0,str(A))\\nfrom wf_data import scenes,NORM\\nresults=A/'results';R=results/'finetune_v1'\\nfinal=json.loads((R/'final_selection.json').read_text());checkpoint=pathlib.Path(final['checkpoint'])\\nassert hashlib.sha256(checkpoint.read_bytes()).hexdigest()==final['checkpoint_sha256']\\nassert not (R/'candidate_cloud_iou_tiles.json').exists(), 'Test already evaluated'\\nns={};exec((A/'cloud_iou_pilot.py').read_text().split('BINS=')[0],ns)\\nm=ns['m'].cuda().eval();torch.set_num_threads(3)\\nw=torch.load(checkpoint,map_location='cpu',weights_only=True)\\nm.load_state_dict({k.removeprefix('network.'):v for k,v in w.items()},strict=True);del w\\n@torch.inference_mode()\\ndef predict(raw):\\n x=(raw.astype(np.float32)-NORM[:,0,None,None])/NORM[:,1,None,None]\\n return (m(torch.from_numpy(x)[None].cuda())[0,1]>0).cpu().numpy()\\ndef cm(p,w,v):return np.array([(p&w&v).sum(),(p&~w&v).sum(),(~p&w&v).sum()],np.int64)\\ndef iou(v):return float(v[0]/sum(v)) if sum(v) else None\\nreference=json.loads((A/'method1_tiles.json').read_text()); refmap={(r['split'],r['scene'],r['r'],r['c']):r for r in reference}\\ninfo={'val':[],'test':[]};natural=[];terrain=[];originals={}\\nfor split in info:\\n for name,s2,gt in scenes(split):\\n  ts=[]\\n  with rasterio.open(s2) as S,rasterio.open(gt) as G:\\n   for r in range(0,G.height-255,512):\\n    for c in range(0,G.width-255,512):\\n     win=Window(c,r,256,256);g=G.read(window=win);v=g[1]>0\\n     if v.mean()<.5:continue\\n     f=float((g[0]==2)[v].mean());ts.append((r,c,g,f));raw=S.read(list(range(1,14)),window=win)\\n     p=predict(raw);w=g[1]==2;cloud=g[0]==2;key=(split,name,r,c);originals[key]=p\\n     counts=cm(p,w,v).tolist()\\n     natural.append(dict(split=split,scene=name,r=r,c=c,cloud_frac=f,all=counts,clear=cm(p,w,v&~cloud).tolist(),under_cloud=cm(p,w,v&cloud).tolist()))\\n     item=dict(refmap[key]);item['cm']={'base':counts};terrain.append(item)\\n  info[split].append(dict(name=name,s2=s2,tiles=ts))\\n  print('CANDIDATE NATURAL',split,name,len(natural),flush=True)\\n if split=='val':\\n  vv={group:pooled_val for group,pooled_val in [(group,iou(np.array([z['all'] for z in natural if filt(z)]).sum(0))) for group,filt in [('all',lambda z:True),('clear',lambda z:z['cloud_frac']<.01),('cloudy',lambda z:z['cloud_frac']>=.1)]]}\\n  assert all(abs(vv[group]-final['chosen']['validation'][group])<1e-5 for group in vv), ('Export validation failed before test',vv)\\nassert set(originals)==set(refmap)\\n(R/'candidate_cloud_iou_tiles.json').write_text(json.dumps(natural))\\n(R/'candidate_method1_tiles.json').write_text(json.dumps(terrain))\\ndef pooled(rs):return iou(np.array([r['all'] for r in rs]).sum(0))\\nval={g:pooled([r for r in natural if r['split']=='val' and f(r)]) for g,f in [('all',lambda r:True),('clear',lambda r:r['cloud_frac']<.01),('cloudy',lambda r:r['cloud_frac']>=.1)]}\\nassert all(abs(val[g]-final['chosen']['validation'][g])<1e-5 for g in val), ('Calibration export mismatch',val,final['chosen']['validation'])\\nimport builtins,runpy\\nfrom unittest.mock import patch\\nreal_open=builtins.open\\nmaps={'cloud_iou_tiles.json':R/'candidate_cloud_iou_tiles.json','method1_tiles.json':R/'candidate_method1_tiles.json'}\\nfor script,outname in [('kill_test.py','kill_test_results.json'),('ndvi_comatch_analyze.py','ndvi_comatch_results.json')]:\\n def redirected(file,mode='r',*args,**kwargs):\\n  if isinstance(file,(str,os.PathLike)):\\n   name=pathlib.Path(file).name\\n   if any(c in mode for c in 'wax+'):\\n    assert name==outname;file=R/('candidate_'+outname)\\n   elif name in maps:file=maps[name]\\n  return real_open(file,mode,*args,**kwargs)\\n with patch('builtins.open',redirected):runpy.run_path(str(A/script),run_name='__main__')\\ncontrols=json.loads((results/'pooled_cloud_noncloud_placebo.json').read_text())\\ncontrolmap={(r['split'],r['scene']):r for r in controls['rows']};rows=[]\\nfor split,scs in info.items():\\n donors=[(s['name'],r,c,g[0]==2) for s in scs for r,c,g,f in s['tiles'] if .1<=f<=.6]\\n for s in scs:\\n  if (split,s['name']) not in controlmap:continue\\n  control=controlmap[(split,s['name'])];own=[g[0]==2 for r,c,g,f in s['tiles'] if f>=.1];clear=[(r,c,g) for r,c,g,f in s['tiles'] if f<.01]\\n  acc={k:np.zeros(3,np.int64) for k in ('original','cloud','placebo')}\\n  with rasterio.open(s['s2']) as S:\\n   for i,(r,c,g) in enumerate(clear):\\n    mask=own[i%len(own)] if own else donors[(i+42)%len(donors)][3]\\n    raw=S.read(list(range(1,14)),window=Window(c,r,256,256)).astype(np.float32);valid=(g[1]>0)&~mask;water=g[1]==2\\n    acc['original']+=cm(originals[(split,s['name'],r,c)],water,valid)\\n    for kind,key in [('cloud','cloud_spectrum'),('placebo','noncloud_spectrum')]:\\n     painted=raw.copy();painted[:,mask]=np.array(control[key],np.float32)[:,None]\\n     acc[kind]+=cm(predict(painted),water,valid)\\n  scores={k:iou(v) for k,v in acc.items()};assert all(v is not None for v in scores.values())\\n  row=dict(split=split,scene=s['name'],n_clear_tiles=len(clear),confusions={k:v.tolist() for k,v in acc.items()},iou=scores)\\n  row['gap_cloud_pts']=100*(scores['original']-scores['cloud']);row['gap_placebo_pts']=100*(scores['original']-scores['placebo']);row['cloud_minus_placebo_pts']=row['gap_cloud_pts']-row['gap_placebo_pts'];rows.append(row)\\n  print('CANDIDATE PASTE',split,s['name'],row['gap_cloud_pts'],flush=True)\\ndef summary(rs,key):\\n g=np.array([r[key] for r in rs]);rng=np.random.default_rng(42);bs=[rng.choice(g,len(g)).mean() for _ in range(5000)]\\n return dict(scenes=len(g),mean_gap_pts=float(g.mean()),ci95=np.percentile(bs,[2.5,97.5]).tolist())\\npaste_summary={sp:{k:summary([r for r in rows if sp=='val_test' or r['split']==sp],k) for k in ('gap_cloud_pts','gap_placebo_pts','cloud_minus_placebo_pts')} for sp in ('val_test','val','test')}\\n(R/'candidate_same_protocol.json').write_text(json.dumps(dict(rule='Same corrected cloud/noncloud spectra, masks and unpainted-pixel scoring as frozen audit; candidate selected only on validation.',rows=rows,summary=paste_summary),indent=2))\\nbase=json.loads((results/'cloud_iou_gpu_tiles.json').read_text())\\ncomparison={sp:{g:{'baseline_iou':pooled([r for r in base if r['split']==sp and f(r)]),'candidate_iou':pooled([r for r in natural if r['split']==sp and f(r)])} for g,f in [('all',lambda r:True),('clear',lambda r:r['cloud_frac']<.01),('cloudy',lambda r:r['cloud_frac']>=.1)]} for sp in ('val','test')}\\npaired={}\\nfor sp in ('val','test'):\\n deltas=[]\\n for name in sorted({r['scene'] for r in natural if r['split']==sp}):\\n  filt=lambda r:r['split']==sp and r['scene']==name\\n  b=pooled([r for r in base if filt(r)]);p=pooled([r for r in natural if filt(r)])\\n  if b is not None and p is not None:deltas.append(dict(scene=name,baseline_iou=b,candidate_iou=p,delta_pts=100*(p-b)))\\n paired[sp]=dict(rows=deltas,summary=summary(deltas,'delta_pts'))\\nreport=dict(selection=final,comparison=comparison,paired_scene_comparison=paired,target_met=comparison['test']['all']['candidate_iou']>=.9,test_reused='Existing test was previously inspected for the original research audit; not a fresh blind benchmark.',no_test_selection=True)\\n(R/'candidate_comparison.json').write_text(json.dumps(report,indent=2))\\nprint('FINAL CANDIDATE COMPARISON',json.dumps(report),flush=True)\\n\")\nfor script in ['calibrate_candidate_v1.py','evaluate_candidate_v1.py']:\n compile((audit/script).read_text(),script,'exec')\n stream([sys.executable,'-u',str(audit/script)])\nprint('FINE-TUNE EVALUATION COMPLETE',flush=True)\n")

CALIBRATION VAL released EMSR260_07CAMPOGALLIANO_DEL_MONIT01_v1
CALIBRATION VAL released EMSR271_02FARKADONA_DEL_v1
CALIBRATION VAL released EMSR274_05TOAMASINA_DEL_v1
CALIBRATION VAL released EMSR279_05ZARAGOZA_DEL_v2
CALIBRATION VAL released EMSR279_11TORTOSA_DEL_v1
CALIBRATION VAL released EMSR280_03FALUN_DEL_MONIT06_v2
CALIBRATION VAL released EMSR293_03OZUN_GRA_v1
CALIBRATION VAL released EMSR312_08CANDON_DEL_MONIT01_v1
CALIBRATION VAL released EMSR329_05CASTIADAS_GRA_v3
CALIBRATION VAL released EMSR358_AOI01_FEP_PRODUCT
CALIBRATION VAL released EMSR427_AOI03_DEL_MONIT02
CALIBRATION VAL released EMSR442_AOI03_DEL_PRODUCT
CALIBRATION VAL released EMSR629_AOI01_DEL_PRODUCT
CALIBRATION VAL released EMSR629_AOI02_DEL_PRODUCT
CALIBRATION VAL released RS2_20161008_Water_Extent_Corail_Pestel
CALIBRATION VAL released ST1_20161014_WaterExtent_BinhDinh_Lake
CALIBRATION VAL best_eligible EMSR260_07CAMPOGALLIANO_DEL_MONIT01_v1
CALIBRATION VAL best_eligible EMSR271_02FARKADONA_DEL_v1
CALIBRATI

In [ ]:
exec("import json,pathlib,zipfile,hashlib,datetime\nfrom google.colab import files\nR=audit/'results'/'finetune_v1'; C=audit/'checkpoints'/'finetune_v1'\ncomparison=json.loads((R/'candidate_comparison.json').read_text()); chosen=comparison['selection']; test=comparison['comparison']['test']; hist=json.loads((R/'history.json').read_text());selection=json.loads((R/'selection.json').read_text())\nlines=['# WF2 weight training and validation selection','',f\"Final test pooled water IoU: **{100*test['all']['candidate_iou']:.2f}%**, released checkpoint **{100*test['all']['baseline_iou']:.2f}%**. 90% target {'met' if comparison['target_met'] else 'not met'}.\",'',f\"Selected source: {chosen['chosen']['source']['name']}, epoch {chosen['chosen']['source']['epoch']}. Validation threshold {chosen['chosen']['threshold']:.2f} encoded as water-output bias shift {chosen['water_bias_shift']:.8f}; exported inference threshold remains 0.5.\",'','| Test tile group | Released IoU | Candidate IoU |','|---|---:|---:|']\nfor group,item in test.items():lines.append(f\"| {group} | {100*item['baseline_iou']:.2f}% | {100*item['candidate_iou']:.2f}% |\")\nlines+=['',f\"Training completed {selection['epochs_completed']} epochs on {len(json.loads((R/'train_index.json').read_text()))} train tiles from 388 scenes. Entire UNet weights were fine-tuned; encoder LR5e-6, decoder LR5e-5, batch8, 512 steps/epoch, maximum16 epochs. Train activations appearing in evaluation were excluded. Full details: config.json and history.json.\",'','Checkpoint and threshold selection used validation only, with clear and cloudy validation IoU each constrained to no more than one point below the released baseline. The final weights were frozen before evaluation. The existing test had been inspected for the original research audit and is not a fresh blind benchmark. The target refers to water intersection-over-union, not overall pixel accuracy.','',f\"Released checkpoint SHA256: {selection['released_checkpoint_sha256']}; verified unchanged. Exported checkpoint SHA256: {chosen['checkpoint_sha256']}.\",'','All reported candidate results are newly written under results/finetune_v1. Composition matching, NDVI matching, pooled scene aggregation and paired cloud/noncloud constant-fill controls were rerun with unchanged protocol. Synthetic fill is a disruption test, not a simulation of physical clouds.']\npaired=comparison['paired_scene_comparison']['test']['summary'];lines+=['',f\"Paired mean test scene IoU change: {paired['mean_gap_pts']:.3f} points, scene bootstrap 95% CI {paired['ci95']} ({paired['scenes']} scenes).\"]\n(R/'FINETUNE_REPORT.md').write_text('\n'.join(lines)+'\n')\npackage=pathlib.Path('/content/worldfloods-finetune-v1.zip')\nwith zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED) as z:\n for p in sorted(R.rglob('*')):\n  if p.is_file():z.write(p,str(p.relative_to(audit.parent)))\n for name in ['train_candidate_v1.py','calibrate_candidate_v1.py','evaluate_candidate_v1.py','FINETUNE_V1_RULE.md','FINETUNE_V1_CALIBRATION_RULE.md']:\n  p=audit/name;z.write(p,str(p.relative_to(audit.parent)))\n for name in ['calibrated_selected_model.pt','best_eligible_model.pt','best_validation_model.pt']:\n  p=C/name\n  if p.exists():z.write(p,str(p.relative_to(audit.parent)))\nprint('FINE-TUNE PACKAGE READY',package,package.stat().st_size,flush=True)\nfiles.download(str(package))\n")

SyntaxError: unterminated string literal (detected at line 9) (<string>, line 9)

In [ ]:
exec("(audit/'pooled_placebo_followup.py').write_text(\"import os, sys, json, pathlib, torch, subprocess\\nimport numpy as np, rasterio\\nfrom rasterio.windows import Window\\naudit=pathlib.Path(__file__).resolve().parent\\nos.chdir(audit)\\nresults=audit/'results'\\nresults.mkdir(exist_ok=True)\\nwf_root=pathlib.Path(os.environ['WF_ROOT'])\\nassert wf_root.exists() and pathlib.Path(os.environ['WF_CKPT']).exists(), 'STOP: missing WF_ROOT or WF_CKPT'\\nsys.path.insert(0,str(audit))\\nfrom wf_data import scenes, NORM\\nmanifest=json.loads((results/'download_manifest.json').read_text())\\nimport numpy as np, rasterio\\nfrom rasterio.windows import Window\\nfrom wf_data import scenes, NORM\\nassert manifest['complete']\\nrule = '''Follow-up registered before new results. Frozen WF2_unetv2_all; 256px grid, stride 512, >=50% valid water labels; clear cf<0.01, donor cf>=0.1; eligible scenes >=200 clear-water pixels. Same-scene donors cycle row/col. Fallback donors are sorted within split with labelled-pixel cf in [0.1,0.6], index (i+42)%K. Corrected cloud fill: exact per-band pooled median over every GT-cloud pixel in kept scene tiles if >=1000 cloud pixels, else exact pooled validation-cloud median. Non-cloud placebo: exact per-band pooled median of valid GT-clear pixels of the same scene, or frozen pooled validation clear pixels if <1000. Identical masks for both fills; constant hard paste; score only unpainted valid pixels. This placebo measures generic constant-fill disruption, not natural cloud. Scene confusion pools, unweighted scene means, 5000 scene bootstrap draws, seed 42; original result unchanged. No tuning. Exact integer histograms compute medians without subsampling.'''\\n(audit/'FOLLOWUP_POOLED_PLACEBO_RULE.md').write_text(rule)\\nnamespace = {}\\nexec((audit/'cloud_iou_pilot.py').read_text().split('BINS=')[0], namespace)\\nmodel = namespace['m'].to('cuda').eval()\\n@torch.inference_mode()\\ndef predict(raw):\\n    x = (raw.astype(np.float32)-NORM[:,0,None,None])/NORM[:,1,None,None]\\n    return (torch.sigmoid(model(torch.from_numpy(x)[None].to('cuda')))[0,1]>0.5).cpu().numpy()\\ndef cm(p,w,v):\\n    return np.array([(p&w&v).sum(),(p&~w&v).sum(),(~p&w&v).sum()],np.int64)\\ndef iou(a):\\n    return float(a[0]/a.sum()) if a.sum() else None\\ndef median_hist(h):\\n    n=int(h[0].sum())\\n    assert n>0\\n    c=np.cumsum(h,axis=1)\\n    return np.array([(np.searchsorted(row,(n-1)//2+1)+np.searchsorted(row,n//2+1))/2 for row in c],np.float32)\\ndef add_hist(h,raw,mask):\\n    for band in range(13):\\n        vals=raw[band,mask]\\n        assert np.all(vals==vals.astype(np.uint16))\\n        h[band] += np.bincount(vals.astype(np.int64),minlength=65536)\\ninfo = {'val':[],'test':[]}\\nfor split in info:\\n    for name,s2,gt in scenes(split):\\n        ts=[]; hc=np.zeros((13,65536),np.int64); hn=hc.copy()\\n        with rasterio.open(s2) as S, rasterio.open(gt) as G:\\n            for r in range(0,G.height-255,512):\\n                for c in range(0,G.width-255,512):\\n                    win=Window(c,r,256,256); g=G.read(window=win); v=g[1]>0\\n                    if v.mean()<0.5: continue\\n                    raw=S.read(list(range(1,14)),window=win)\\n                    f=float((g[0]==2)[v].mean())\\n                    ts.append((r,c,g,f))\\n                    add_hist(hc,raw,g[0]==2)\\n                    add_hist(hn,raw,(g[0]==1)&v)\\n        info[split].append(dict(name=name,s2=s2,tiles=ts,hc=hc,hn=hn))\\n        print('EXACT SCAN',split,name,len(ts),flush=True)\\nfrozen_cloud=median_hist(sum(s['hc'] for s in info['val']))\\nfrozen_clear=median_hist(sum(s['hn'] for s in info['val']))\\nrows=[]\\nfor split,scs in info.items():\\n    donors=[(s['name'],r,c,g[0]==2) for s in scs for r,c,g,f in s['tiles'] if 0.1<=f<=0.6]\\n    for s in scs:\\n        clear=[(r,c,g) for r,c,g,f in s['tiles'] if f<0.01]\\n        cw=sum(int(((g[0]==1)&(g[1]==2)).sum()) for r,c,g in clear)\\n        if cw<200: continue\\n        own=[g[0]==2 for r,c,g,f in s['tiles'] if f>=0.1]\\n        assert own or donors\\n        cloud=median_hist(s['hc']) if s['hc'][0].sum()>=1000 else frozen_cloud\\n        noncloud=median_hist(s['hn']) if s['hn'][0].sum()>=1000 else frozen_clear\\n        acc={k:np.zeros(3,np.int64) for k in ('original','cloud','placebo')}\\n        with rasterio.open(s['s2']) as S:\\n            for i,(r,c,g) in enumerate(clear):\\n                mask=own[i%len(own)] if own else donors[(i+42)%len(donors)][3]\\n                raw=S.read(list(range(1,14)),window=Window(c,r,256,256)).astype(np.float32)\\n                valid=(g[1]>0)&~mask; water=g[1]==2\\n                acc['original']+=cm(predict(raw),water,valid)\\n                for kind,spectrum in [('cloud',cloud),('placebo',noncloud)]:\\n                    painted=raw.copy(); painted[:,mask]=spectrum[:,None]\\n                    acc[kind]+=cm(predict(painted),water,valid)\\n        scores={k:iou(v) for k,v in acc.items()}\\n        row=dict(split=split,scene=s['name'],n_clear_tiles=len(clear),clear_water_px=cw,own_mask=bool(own),own_cloud_spectrum=bool(s['hc'][0].sum()>=1000),cloud_spectrum=cloud.tolist(),noncloud_spectrum=noncloud.tolist(),confusions={k:v.tolist() for k,v in acc.items()},iou=scores)\\n        row['gap_cloud_pts']=100*(scores['original']-scores['cloud'])\\n        row['gap_placebo_pts']=100*(scores['original']-scores['placebo'])\\n        row['cloud_minus_placebo_pts']=row['gap_cloud_pts']-row['gap_placebo_pts']\\n        rows.append(row)\\n        (results/'pooled_placebo_progress.json').write_text(json.dumps(rows,indent=2))\\n        print('POOLED PLACEBO',split,s['name'],row['gap_cloud_pts'],row['gap_placebo_pts'],flush=True)\\ndef summary(rs,key):\\n    g=np.array([r[key] for r in rs],float)\\n    rng=np.random.default_rng(42)\\n    bs=np.array([rng.choice(g,len(g)).mean() for _ in range(5000)])\\n    return dict(scenes=len(g),mean_gap_pts=float(g.mean()),ci95=np.percentile(bs,[2.5,97.5]).tolist())\\nsummary_result={sp:{k:summary([r for r in rows if sp=='val_test' or r['split']==sp],k) for k in ('gap_cloud_pts','gap_placebo_pts','cloud_minus_placebo_pts')} for sp in ('val_test','val','test')}\\noutput=dict(rule=rule,seed=42,device='cuda',frozen_val_cloud_spectrum=frozen_cloud.tolist(),frozen_val_clear_spectrum=frozen_clear.tolist(),rows=rows,summary=summary_result)\\n(results/'pooled_cloud_noncloud_placebo.json').write_text(json.dumps(output,indent=2))\\nprint('PASTE SUMMARY',json.dumps(summary_result),flush=True)\\n\")\n(audit/'jrc_brightness_followup.py').write_text(\"import os, sys, json, pathlib, torch, subprocess\\nimport numpy as np, rasterio\\nfrom rasterio.windows import Window\\naudit=pathlib.Path(__file__).resolve().parent\\nos.chdir(audit)\\nresults=audit/'results'\\nresults.mkdir(exist_ok=True)\\nwf_root=pathlib.Path(os.environ['WF_ROOT'])\\nassert wf_root.exists() and pathlib.Path(os.environ['WF_CKPT']).exists(), 'STOP: missing WF_ROOT or WF_CKPT'\\nsys.path.insert(0,str(audit))\\nfrom wf_data import scenes, NORM\\nmanifest=json.loads((results/'download_manifest.json').read_text())\\nnamespace = {}\\nexec((audit/'cloud_iou_pilot.py').read_text().split('BINS=')[0], namespace)\\nmodel = namespace['m'].to('cuda').eval()\\n@torch.inference_mode()\\ndef predict(raw):\\n    x = (raw.astype(np.float32)-NORM[:,0,None,None])/NORM[:,1,None,None]\\n    return (torch.sigmoid(model(torch.from_numpy(x)[None].to('cuda')))[0,1]>0.5).cpu().numpy()\\ndef cm(p,w,v):\\n    return np.array([(p&w&v).sum(),(p&~w&v).sum(),(~p&w&v).sum()],np.int64)\\ndef iou(a):\\n    return float(a[0]/a.sum()) if a.sum() else None\\ndef median_hist(h):\\n    n=int(h[0].sum())\\n    assert n>0\\n    c=np.cumsum(h,axis=1)\\n    return np.array([(np.searchsorted(row,(n-1)//2+1)+np.searchsorted(row,n//2+1))/2 for row in c],np.float32)\\ndef add_hist(h,raw,mask):\\n    for band in range(13):\\n        vals=raw[band,mask]\\n        assert np.all(vals==vals.astype(np.uint16))\\n        h[band] += np.bincount(vals.astype(np.int64),minlength=65536)\\ninfo={'val':[],'test':[]}\\nfor split in info:\\n    for name,s2,gt in scenes(split):\\n        ts=[]\\n        with rasterio.open(gt) as G:\\n            for r in range(0,G.height-255,512):\\n                for c in range(0,G.width-255,512):\\n                    g=G.read(window=Window(c,r,256,256)); v=g[1]>0\\n                    if v.mean()<.5: continue\\n                    ts.append((r,c,g,float((g[0]==2)[v].mean())))\\n        info[split].append(dict(name=name,s2=s2,tiles=ts))\\nrule_strata = '''Registered before census/strata results. Same frozen WF2 baseline and 256px/512 grid. Brightness=sqrt(B2^2+B3^2+B4^2)/10000, per official ml4floods get_brightness at commit 192426c7286315b59a2f52c7a9cdebe589242de2; cuts .3 and .35; count GT-cloud pixels above/below each cut, with water counts, on valid water-label pixels. JRC strata: permanent JRC==3; nonpermanent JRC in {1,2}; unknown JRC==0 excluded from strata but counted. Compute binary water IoU within each spatial stratum, confusion pooled within scenes; report unweighted scene mean and 5000 scene-bootstrap CI seed42, plus pooled IoU explicitly as secondary. Nonpermanent water is a JRC-based proxy, not independently confirmed flood water. Also report raw separate per-scene IoU for Campo Galliano and Ituango South; within-scene tile bootstrap 5000 draws seed42 is descriptive and not an event interval. Never add these to existing gap means.'''\\n(audit/'JRC_BRIGHTNESS_SCENE_RULE.md').write_text(rule_strata)\\nstrata_rows=[]; census_rows=[]; separate=[]\\nfor split,scs in info.items():\\n    for s in scs:\\n        name=s['name']; jrc=wf_root/split/'PERMANENTWATERJRC'/(name+'.tif')\\n        assert jrc.exists(), 'STOP missing JRC raster'\\n        counts={k:np.zeros(3,np.int64) for k in ('permanent','nonpermanent','all')}\\n        buckets={str(cut):{k:dict(pixels=0,water_pixels=0) for k in ('gt_cloud_le_cut','gt_cloud_gt_cut','gt_clear_le_cut','gt_clear_gt_cut')} for cut in (0.3,0.35)}\\n        jrc_counts={}; tile_cm=[]\\n        with rasterio.open(s['s2']) as S, rasterio.open(jrc) as J:\\n            assert S.shape==J.shape and S.transform==J.transform and S.crs==J.crs, 'STOP: JRC alignment differs'\\n            for r,c,g,f in s['tiles']:\\n                win=Window(c,r,256,256)\\n                raw=S.read(list(range(1,14)),window=win).astype(np.float32)\\n                j=J.read(1,window=win); v=g[1]>0; wt=g[1]==2\\n                pred=predict(raw)\\n                counts['all']+=cm(pred,wt,v)\\n                counts['permanent']+=cm(pred,wt,v&(j==3))\\n                counts['nonpermanent']+=cm(pred,wt,v&np.isin(j,[1,2]))\\n                tile_cm.append(cm(pred,wt,v).tolist())\\n                for label,n in zip(*np.unique(j[v],return_counts=True)):\\n                    jrc_counts[str(int(label))]=jrc_counts.get(str(int(label)),0)+int(n)\\n                brightness=np.sqrt(np.sum(raw[[1,2,3]]**2,axis=0))/10000.\\n                for cut in (0.3,0.35):\\n                    for cloud_code,prefix in [(2,'gt_cloud'),(1,'gt_clear')]:\\n                        for side,bright in [('le_cut',brightness<=cut),('gt_cut',brightness>cut)]:\\n                            mask=v&(g[0]==cloud_code)&bright\\n                            b=buckets[str(cut)][prefix+'_'+side]\\n                            b['pixels']+=int(mask.sum()); b['water_pixels']+=int((mask&wt).sum())\\n        strata_rows.append(dict(split=split,scene=name,n_tiles=len(s['tiles']),jrc_pixel_counts=jrc_counts,confusions={k:v.tolist() for k,v in counts.items()},iou={k:iou(v) for k,v in counts.items()}))\\n        census_rows.append(dict(split=split,scene=name,cuts=buckets))\\n        if name.startswith('EMSR260_07CAMPOGALLIANO') or name.startswith('EMSR286_09ITUANGOSOUTH'):\\n            rng=np.random.default_rng(42); mat=np.array(tile_cm); values=[]\\n            for _ in range(5000):\\n                vv=iou(mat[rng.integers(0,len(mat),len(mat))].sum(0))\\n                if vv is not None: values.append(vv)\\n            separate.append(dict(split=split,scene=name,tiles=len(mat),iou=iou(counts['all']),ci95_tile_bootstrap=np.percentile(values,[2.5,97.5]).tolist() if values else None,uncertainty='within-scene tile bootstrap; descriptive; not event uncertainty',confusion=counts['all'].tolist()))\\n        print('JRC CENSUS',split,name,flush=True)\\ndef stratum_summary(rs,kind):\\n    vals=[r['iou'][kind] for r in rs if r['iou'][kind] is not None and sum(r['confusions'][kind][::2])>=200]\\n    rng=np.random.default_rng(42)\\n    ci=np.percentile([np.mean(rng.choice(vals,len(vals))) for _ in range(5000)],[2.5,97.5]).tolist() if vals else None\\n    pooled=np.array([r['confusions'][kind] for r in rs]).sum(0)\\n    return dict(scenes=len(vals),scene_gate='>=200 GT-water pixels in stratum',scene_mean_iou=float(np.mean(vals)) if vals else None,ci95_scene_bootstrap=ci,pooled_iou_secondary=iou(pooled),pooled_confusion=pooled.tolist())\\nstrata_summary={split:{k:stratum_summary([r for r in strata_rows if split=='val_test' or r['split']==split],k) for k in ('permanent','nonpermanent','all')} for split in ('val_test','val','test')}\\ncensus_total={str(cut):{k:{field:sum(r['cuts'][str(cut)][k][field] for r in census_rows) for field in ('pixels','water_pixels')} for k in ('gt_cloud_le_cut','gt_cloud_gt_cut','gt_clear_le_cut','gt_clear_gt_cut')} for cut in (.3,.35)}\\n(results/'permanent_nonpermanent_water.json').write_text(json.dumps(dict(rule=rule_strata,rows=strata_rows,summary=strata_summary),indent=2))\\n(results/'brightness_census.json').write_text(json.dumps(dict(rule=rule_strata,rows=census_rows,total=census_total),indent=2))\\n(results/'separate_scene_measurements.json').write_text(json.dumps(dict(rule=rule_strata,scenes=separate),indent=2))\\nprint('STRATA SUMMARY',json.dumps(strata_summary),flush=True)\\nprint('BRIGHTNESS CENSUS',json.dumps(census_total),flush=True)\\nprint('SEPARATE SCENES',json.dumps(separate),flush=True)\\n\")\n(audit/'frozen_gpu_reproduce.py').write_text(\"import os, sys, json, pathlib, torch, subprocess\\nimport numpy as np, rasterio\\nfrom rasterio.windows import Window\\naudit=pathlib.Path(__file__).resolve().parent\\nos.chdir(audit)\\nresults=audit/'results'\\nresults.mkdir(exist_ok=True)\\nwf_root=pathlib.Path(os.environ['WF_ROOT'])\\nassert wf_root.exists() and pathlib.Path(os.environ['WF_CKPT']).exists(), 'STOP: missing WF_ROOT or WF_CKPT'\\nsys.path.insert(0,str(audit))\\nfrom wf_data import scenes, NORM\\nmanifest=json.loads((results/'download_manifest.json').read_text())\\nassert manifest['complete'], 'STOP: missing download'\\nsubprocess.run([sys.executable,'-m','pip','install','-q','rasterio'],check=True)\\nsys.path.insert(0,str(audit))\\nimport numpy as np, rasterio\\nfrom wf_data import scenes\\nprint('SCENES', {sp:len(scenes(sp)) for sp in ('val','test')},flush=True)\\nassert len(scenes('val'))==16 and len(scenes('test'))==18\\nos.environ['OUT'] = 'results/cloud_iou_gpu'\\nsource = (audit/'cloud_iou_pilot.py').read_text()\\nsource = source.replace('m.eval()', \\\"m.eval(); m.to('cuda')\\\")\\nsource = source.replace('m(torch.from_numpy(x)[None])', \\\"m(torch.from_numpy(x)[None].to('cuda'))\\\")\\nsource = source.replace('[0].numpy()', '[0].cpu().numpy()')\\nsaved_argv = sys.argv[:]\\nsys.argv = ['cloud_iou_pilot.py','test','val']\\ntry:\\n    exec(compile(source,'cloud_iou_pilot_gpu','exec'),{'__name__':'__main__'})\\nfinally:\\n    sys.argv = saved_argv\\nnew = json.loads((results/'cloud_iou_gpu_tiles.json').read_text())\\nref = json.loads((audit/'cloud_iou_tiles.json').read_text())\\nkey = lambda x:(x['split'],x['scene'],x['r'],x['c'])\\na,b = {key(x):x for x in new},{key(x):x for x in ref}\\ncomparison = {'tiles':len(a),'reference_tiles':len(b),'same_tile_keys':set(a)==set(b)}\\nif comparison['same_tile_keys']:\\n    changes = [sum(abs(a[k][v][i]-b[k][v][i]) for v in ('all','clear','under_cloud') for i in range(3)) for k in a]\\n    comparison['tiles_with_count_differences'] = sum(x>0 for x in changes)\\n    comparison['max_absolute_count_difference'] = max(changes)\\n(results/'gpu_inference_comparison.json').write_text(json.dumps(comparison,indent=2))\\nprint('INFERENCE COMPARISON',json.dumps(comparison),flush=True)\")\n(audit/'download_inputs.py').write_text(\"\\\"\\\"\\\"Download only public validation/test audit inputs; stop on any failed download.\\\"\\\"\\\"\\nimport os,json,pathlib,urllib.request,urllib.parse,shutil,hashlib\\nroot=pathlib.Path(__file__).resolve().parent\\nresults=root/'results'; results.mkdir(exist_ok=True)\\nwf_root=pathlib.Path(os.environ['WF_ROOT'])\\nwf_ckpt=pathlib.Path(os.environ['WF_CKPT'])\\ndef meta(url):\\n    with urllib.request.urlopen(url,timeout=120) as r: return json.load(r)\\ndef fetch(repo,revision,item,target):\\n    target.parent.mkdir(parents=True,exist_ok=True)\\n    if not target.exists() or target.stat().st_size!=item['size']:\\n        with urllib.request.urlopen(repo+'/resolve/'+revision+'/'+urllib.parse.quote(item['rfilename'],safe='/'),timeout=120) as src,target.open('wb') as dst: shutil.copyfileobj(src,dst)\\n    assert target.stat().st_size==item['size'], 'Incomplete download'\\n    h=hashlib.sha256()\\n    with target.open('rb') as f:\\n        for chunk in iter(lambda:f.read(8388608),b''): h.update(chunk)\\n    if item.get('lfs',{}).get('sha256'): assert h.hexdigest()==item['lfs']['sha256']\\n    return dict(path=str(target),bytes=target.stat().st_size,sha256=h.hexdigest())\\ndataset_revision='1f3faa2989e69930ac31d5c0a79fd224461123f8'\\nmodel_revision='3f52941fbb01b6f66eed71449b7586d6e168e035'\\ndata=meta('https://huggingface.co/api/datasets/isp-uv-es/WorldFloodsv2/revision/'+dataset_revision+'?blobs=true')\\nmodel=meta('https://huggingface.co/api/models/isp-uv-es/ml4floods/revision/'+model_revision+'?blobs=true')\\nselected=[x for x in data['siblings'] if x['rfilename'].split('/')[0] in ('val','test') and x['rfilename'].split('/')[1] in ('S2','gt','PERMANENTWATERJRC') and x['rfilename'].endswith('.tif')]\\nmanifest=dict(dataset_sha=data['sha'],checkpoint_sha=model['sha'],files=[],complete=False)\\ntry:\\n    checkpoint=next(x for x in model['siblings'] if x['rfilename']=='models/WF2_unetv2_all/model.pt')\\n    manifest['files'].append(fetch('https://huggingface.co/isp-uv-es/ml4floods',model['sha'],checkpoint,wf_ckpt))\\n    for i,item in enumerate(selected):\\n        manifest['files'].append(fetch('https://huggingface.co/datasets/isp-uv-es/WorldFloodsv2',data['sha'],item,wf_root/item['rfilename']))\\n        print(i+1,len(selected),item['rfilename'],flush=True)\\n    manifest['complete']=True\\nfinally:\\n    (results/'download_manifest.json').write_text(json.dumps(manifest,indent=2))\\nprid='ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL-Sen1Floods11'\\nprrev='91ce9d38086a80b078a192b374df758b8855b732'\\npr=meta('https://huggingface.co/api/models/'+prid+'/revision/'+prrev+'?blobs=true')\\npm=dict(model_id=prid,revision=pr['sha'],files=[],complete=False)\\ntry:\\n    for name in ('config.yaml','requirements.txt','Prithvi-EO-V2-300M-TL-Sen1Floods11.pt'):\\n        item=next(x for x in pr['siblings'] if x['rfilename']==name)\\n        pm['files'].append(fetch('https://huggingface.co/'+prid,pr['sha'],item,root/'prithvi_assets'/name))\\n    pm['complete']=True\\nfinally:\\n    (results/'prithvi_download_manifest.json').write_text(json.dumps(pm,indent=2))\\nprint('All required downloads verified')\\n\")\nassert (results/'prithvi_same_protocol.json').exists(), 'Prithvi evaluation has not completed'\nimport zipfile, platform\nassert not subprocess.run(['git','diff','--name-only'],cwd=repo,capture_output=True,text=True,check=True).stdout.strip(), 'Tracked files changed'\nwf=json.loads((results/'pooled_cloud_noncloud_placebo.json').read_text())\npr=json.loads((results/'prithvi_same_protocol.json').read_text())\ngpu=json.loads((results/'gpu_inference_comparison.json').read_text())\nassert len(wf['rows'])==len(pr['rows'])==32\nassert gpu['tiles']==1291 and gpu['tiles_with_count_differences']==0\nassert all(x['exact_json_match'] for x in json.loads((results/'reproduction_checks.json').read_text()))\nprovenance=dict(repo_commit=subprocess.run(['git','rev-parse','HEAD'],cwd=repo,capture_output=True,text=True,check=True).stdout.strip(),python=platform.python_version(),numpy=np.__version__,torch=torch.__version__,gpu=torch.cuda.get_device_name(0),dataset_revision=json.loads((results/'download_manifest.json').read_text())['dataset_sha'],prithvi_revision=json.loads((results/'prithvi_download_manifest.json').read_text())['revision'],tracked_files_unchanged=True)\n(results/'runtime_provenance.json').write_text(json.dumps(provenance,indent=2))\n(results/'prithvi_requirements.txt').write_text(subprocess.run(['uv','pip','freeze','--python',prithvi_python],capture_output=True,text=True,check=True).stdout)\nwf_tiles=json.loads((results/'cloud_iou_gpu_tiles.json').read_text())\npr_tiles=json.loads((results/'prithvi_cloud_iou_tiles.json').read_text())\ndef paired_model_summary(split,group):\n    def include(row):\n        return (split=='val_test' or row['split']==split) and (group=='all' or (group=='clear' and row['cloud_frac']<.01) or (group=='cloudy' and row['cloud_frac']>=.1))\n    pools=[]\n    for rows in (wf_tiles,pr_tiles):\n        by={}\n        for row in rows:\n            if include(row):\n                key=(row['split'],row['scene']); by[key]=by.get(key,np.zeros(3,np.int64))+np.array(row['all'])\n        pools.append(by)\n    values=[]; per_scene=[]\n    for key in sorted(set(pools[0])&set(pools[1])):\n        a,b=pools[0][key],pools[1][key]\n        if a[0]+a[2]<200 or b[0]+b[2]<200: continue\n        ai=float(a[0]/a.sum()); bi=float(b[0]/b.sum()); delta=100*(bi-ai)\n        values.append(delta); per_scene.append(dict(split=key[0],scene=key[1],wf2_iou=ai,prithvi_iou=bi,prithvi_minus_wf2_pts=delta))\n    rng=np.random.default_rng(42)\n    ci=np.percentile([rng.choice(values,len(values)).mean() for _ in range(5000)],[2.5,97.5]).tolist() if values else None\n    return dict(scenes=len(values),mean_prithvi_minus_wf2_pts=float(np.mean(values)) if values else None,ci95=ci,per_scene=per_scene)\npaired_comparison={sp:{g:paired_model_summary(sp,g) for g in ('all','clear','cloudy')} for sp in ('val_test','val','test')}\n(results/'paired_model_accuracy_comparison.json').write_text(json.dumps(dict(status='descriptive secondary comparison; no parameter tuning',scene_gate='>=200 GT-water pixels in selected tile group',aggregator='unweighted scene mean of paired accuracy differences',seed=42,n_bootstrap=5000,comparisons=paired_comparison),indent=2))\nprint('PAIRED ACCURACY COMPARISON',json.dumps({sp:{g:{k:v for k,v in s.items() if k!='per_scene'} for g,s in ss.items()} for sp,ss in paired_comparison.items()}),flush=True)\ndef fmt(s): return str(round(s['mean_gap_pts'],3))+' ['+', '.join(str(round(x,3)) for x in s['ci95'])+']'\nlines=['# WorldFloods controlled audit: execution report','','## Completed work','','- Downloaded 102 validation/test rasters (about 5.24 GiB), WF2 weights and Prithvi weights; checksums and revisions are saved.','- Recomputed the three required analyses: exact JSON matches.','- Reran WF2 inference on all 1,291 tiles and 34 scenes: every confusion count matched.','- Ran corrected pooled-median and non-cloud paste controls, JRC water strata, brightness census, two separate scene checks and frozen Prithvi evaluation.','','## Constructed paste controls','','IoU drops in percentage points. Unweighted scene mean; 5,000 scene-bootstrap draws, seed42; 95% intervals. Same masks and scoring surface for both fills.','','| Model | Cloud fill | Non-cloud fill | Paired cloud minus placebo |','|---|---:|---:|---:|']\nfor label,data in [('WF2_UNet',wf),('Prithvi_Sen1Floods11',pr)]:\n    s=data['summary']['val_test']; lines.append('| '+label+' | '+fmt(s['gap_cloud_pts'])+' | '+fmt(s['gap_placebo_pts'])+' | '+fmt(s['cloud_minus_placebo_pts'])+' |')\nlines+=['','The corrected WF2 run uses exact pooled cloud-pixel medians and the written fallback donor fraction filter. The committed implementation sampled pixels, took a median of scene medians for the frozen fallback, and disabled the fallback donor filter with an always-true condition. This is a follow-up correction; it does not replace the committed 5.59-point result. The placebo rule was fixed before computing the result.','','The paste is a constructed constant fill on clear tiles, scored on unpainted labelled pixels. These numbers do not establish natural-cloud causality.','','## Natural composition controls','','| Model | Unmatched gap | NDVI co-match gap |','|---|---:|---:|']\nfor label,pre in [('WF2_UNet',''),('Prithvi_Sen1Floods11','prithvi_')]:\n    k=json.loads((results/(pre+'kill_test_results.json')).read_text())['test_val']['summary']\n    n=json.loads((results/(pre+'ndvi_comatch_results.json')).read_text())['ndvi_comatch']\n    def inter(mean,ci): return str(round(mean,3))+' ['+', '.join(str(round(x,3)) for x in ci)+']'\n    lines.append('| '+label+' | '+inter(k['mean_gap_pts'],k['ci95'])+' (n='+str(k['scenes'])+') | '+inter(n['mean_gap'],n['ci95'])+' (n='+str(n['n_scenes'])+') |')\nlines+=['','Both models use unchanged terrain metadata, within-scene matching and named scene aggregation. Pixel-pooled comparisons are secondary and saved separately. A smaller paste drop alone does not establish a better model.','','## JRC strata, brightness and separate scenes','','JRC==3 is permanent; codes1/2 are known nonpermanent; code0 is unknown and excluded from these stratum IoUs. Nonpermanent is a proxy, not independently confirmed flood water. These are binary IoUs within spatial strata, not full-image multiclass flood-type IoUs.','']\nstrata=json.loads((results/'permanent_nonpermanent_water.json').read_text())\ncoverage={}\nfor row in strata['rows']:\n    for key,count in row['jrc_pixel_counts'].items(): coverage[key]=coverage.get(key,0)+count\nlines.append('JRC pixel counts by code: '+json.dumps(coverage)+'.')\nfor kind,s in strata['summary']['val_test'].items():\n    lines.append('- '+kind+': scene-mean IoU '+str(round(s['scene_mean_iou'],5))+'; n='+str(s['scenes'])+'; scene-bootstrap CI '+str(s['ci95_scene_bootstrap'])+'.')\nlines+=['','Brightness is sqrt(B2 squared+B3 squared+B4 squared)/10000, following official ml4floods get_brightness at commit192426c7286315b59a2f52c7a9cdebe589242de2. Counts at0.3/0.35 are saved in brightness_census.json. No arbitrary DN range is called thin cloud.','','The following scenes remain separate. Their intervals use 5,000 within-scene tile-bootstrap draws and are descriptive, not event-level uncertainty:','']\nfor row in json.loads((results/'separate_scene_measurements.json').read_text())['scenes']:\n    lines.append('- '+row['scene']+': '+str(row['tiles'])+' tiles; IoU '+str(round(row['iou'],5))+'; interval '+str(row['ci95_tile_bootstrap'])+'.')\nlines+=['','## Reproduction and limitations','','- Frozen checkpoints; no additional training, threshold tuning, paper changes or replacement of committed JSON.','- Prithvi uses six official bands, reflectance scaling and Sen1Floods11 means/stds, native256px inputs, no metadata as configured, and two-class argmax.','- Prithvi was trained on Sen1Floods11 and is evaluated across datasets; differences cannot be attributed solely to cloud robustness.','- Prithvi runs in isolated Python3.11 with TerraTorch0.99.8 because the current Colab Python3.13 runtime conflicts with its old NumPy pin. Exact installed packages are included.','- New intervals are scene-bootstrap intervals except explicitly labelled within-scene checks; they are not activation/event-clustered.','- Rule files and standalone follow-up scripts are included. Set WF_ROOT and WF_CKPT to existing downloaded inputs before rerunning.','- Source commit: '+provenance['repo_commit'],'- Dataset revision: '+provenance['dataset_revision'],'- Prithvi revision: '+provenance['prithvi_revision'],'','## Primary sources','','- https://github.com/KanakMalpani/Reproducing-research-','- https://huggingface.co/datasets/isp-uv-es/WorldFloodsv2','- https://huggingface.co/isp-uv-es/ml4floods','- https://www.nature.com/articles/s41598-023-47595-7','- https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL-Sen1Floods11','']\nlines+=['','## Descriptive paired model accuracy','','Unweighted scene means of Prithvi minus WF2 IoU, points; 5,000 scene-bootstrap draws seed42. Gate: at least200 GT-water pixels per scene/group. No tuning.','']\nfor sp,groups in paired_comparison.items():\n    for group,s in groups.items():\n        lines.append('- '+sp+'/'+group+': '+str(s['mean_prithvi_minus_wf2_pts'])+' points; CI '+str(s['ci95'])+'; n='+str(s['scenes'])+'.')\n(results/'RUN_REPORT.md').write_text(chr(10).join(lines))\nfor filename in ('pooled_placebo_followup.py','jrc_brightness_followup.py','prithvi_evaluate.py'):\n    compile((audit/filename).read_text(),filename,'exec')\nbundle=pathlib.Path('/content/worldfloods-audit-results.zip')\nwith zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED) as z:\n    for p in audit.iterdir():\n        if p.is_file() and p.suffix.lower() in ('.py','.md','.json'): z.write(p,'clear-cloudy-gap-audit/'+p.name)\n    for p in results.iterdir():\n        if p.is_file(): z.write(p,'clear-cloudy-gap-audit/results/'+p.name)\nprint('AUDIT EXPORT READY',str(bundle),bundle.stat().st_size,flush=True)\nprint('FINAL CONTROL SUMMARY',json.dumps({'WF2':wf['summary']['val_test'],'Prithvi':pr['summary']['val_test']}),flush=True)\nfrom google.colab import files\nfiles.download(str(bundle))\n")

PAIRED ACCURACY COMPARISON {"val_test": {"all": {"scenes": 33, "mean_prithvi_minus_wf2_pts": -5.898772959894051, "ci95": [-11.039671311877484, -1.4201558036081283]}, "clear": {"scenes": 32, "mean_prithvi_minus_wf2_pts": -6.254473066997678, "ci95": [-12.06356465636519, -1.2679073656094058]}, "cloudy": {"scenes": 8, "mean_prithvi_minus_wf2_pts": -15.2168790085341, "ci95": [-29.103670523277284, -3.135720262173285]}}, "val": {"all": {"scenes": 15, "mean_prithvi_minus_wf2_pts": -6.545567678036232, "ci95": [-16.225892938022213, 1.7774723594793578]}, "clear": {"scenes": 15, "mean_prithvi_minus_wf2_pts": -5.988145185908343, "ci95": [-15.71187048251037, 2.2892512956541218]}, "cloudy": {"scenes": 3, "mean_prithvi_minus_wf2_pts": -8.577032041216487, "ci95": [-9.511139246853329, -7.533390429460692]}}, "test": {"all": {"scenes": 18, "mean_prithvi_minus_wf2_pts": -5.359777361442234, "ci95": [-9.883572165763796, -1.2864336063656687]}, "clear": {"scenes": 17, "mean_prithvi_minus_wf2_pts": -6.489468256

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>